# Parkinson AI — Colab GPU pipeline (compact checkpoints)
## WideResNet50-2 + Focal Loss → DeepRBF → combined inference

This notebook keeps intermediate WideResNet checkpoints resumable, then stores each completed fold without the large optimizer/RNG state. The exact model weights and provenance metadata are preserved. This substantially reduces Drive usage; the storage preflight estimates the additional space required and stops before training if the actual free space is insufficient.


## 0. Connect to Colab, mount the shared Drive, and prepare the code/data

### VS Code connection

1. Install the official **Google Colab** extension in VS Code.
2. Open this `.ipynb` file.
3. Click **Select Kernel** → **Colab** → choose a Colab runtime.
4. Sign in to the Google account that has access to the team's `ParkinsonResearch` Drive folder.
5. Select a runtime with a **GPU**.

The first code cell checks CUDA and stops if the runtime is not a Colab GPU.

### Why the code comes from GitHub

The previous `parkinson_training_code_v2.zip` was made for the baseline notebook. To avoid silently using an older code snapshot, this notebook checks out the current DeepRBF-only source at a fixed Git commit. The shared dataset remains the exact versioned ZIP already used by the team.

The dataset is copied from Drive to Colab's temporary disk, checksum-verified, safely extracted, and every packaged file is verified. Model checkpoints and reports are saved back to Drive so they survive a Colab runtime reset.

In [1]:
from pathlib import Path
import ast
import hashlib
import json
import os
import re
import shutil
import stat
import subprocess
import sys
import zipfile

# This notebook is intended for the official Google Colab kernel in VS Code.
try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError(
        "Use Select Kernel -> Colab in VS Code, sign in, and connect to a GPU runtime."
    ) from exc

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No Colab CUDA GPU is attached. Select a GPU runtime and rerun this cell. "
        "No training has started."
    )

drive.mount("/content/drive")

# ---------- Shared Drive project discovery ----------
DRIVE_ROOT = Path("/content/drive")
SHORTCUT_ROOT = DRIVE_ROOT / ".shortcut-targets-by-id"
ARCHIVE_RELATIVE = Path("datasets") / "parkinson_dataset_v1.zip"

# IMPORTANT:
# Colab sometimes allows access to a known shortcut path even when
# SHORTCUT_ROOT.iterdir() returns an empty list. The ID below is the
# exact shortcut path confirmed in the user's Colab diagnostic output.
KNOWN_SHARED_PROJECT = (
    DRIVE_ROOT
    / ".shortcut-targets-by-id"
    / "1UGKWBfKn2fvGjBr39P1v9ndyA5_8A2Gs"
    / "ParkinsonResearch"
)
DIRECT_PROJECT = DRIVE_ROOT / "MyDrive" / "ParkinsonResearch"

def is_valid_project_folder(candidate: Path) -> bool:
    return candidate.is_dir() and (candidate / ARCHIVE_RELATIVE).is_file()

# Prefer the exact confirmed shortcut path. Do not depend on listing
# .shortcut-targets-by-id, because Colab may expose it as empty.
if is_valid_project_folder(KNOWN_SHARED_PROJECT):
    DRIVE_DIR = KNOWN_SHARED_PROJECT
elif is_valid_project_folder(DIRECT_PROJECT):
    DRIVE_DIR = DIRECT_PROJECT
else:
    # Fallback discovery for users whose shortcut ID is different.
    project_matches = []
    if SHORTCUT_ROOT.is_dir():
        try:
            for shortcut in SHORTCUT_ROOT.iterdir():
                candidate = shortcut / "ParkinsonResearch"
                if is_valid_project_folder(candidate):
                    project_matches.append(candidate)
        except OSError as exc:
            print("Shortcut directory could not be enumerated:", repr(exc))

    # Deduplicate without resolving Drive/FUSE shortcut paths.
    unique_matches = {str(candidate): candidate for candidate in project_matches}

    if len(unique_matches) == 1:
        DRIVE_DIR = next(iter(unique_matches.values()))
    else:
        known_contents = (
            [p.name for p in KNOWN_SHARED_PROJECT.iterdir()]
            if KNOWN_SHARED_PROJECT.is_dir()
            else "project folder not accessible"
        )
        dataset_contents = (
            [p.name for p in (KNOWN_SHARED_PROJECT / "datasets").iterdir()]
            if (KNOWN_SHARED_PROJECT / "datasets").is_dir()
            else "datasets folder not accessible"
        )
        raise FileNotFoundError(
            "Could not locate the project archive.\n"
            f"Known project path: {KNOWN_SHARED_PROJECT}\n"
            f"Known project accessible: {KNOWN_SHARED_PROJECT.is_dir()}\n"
            f"Project contents: {known_contents}\n"
            f"datasets contents: {dataset_contents}\n"
            f"Fallback candidates: {list(unique_matches.values())}\n"
            "Expected archive: ParkinsonResearch/datasets/parkinson_dataset_v1.zip"
        )

SOURCE_ZIP = DRIVE_DIR / ARCHIVE_RELATIVE
if not SOURCE_ZIP.is_file():
    raise FileNotFoundError(
        f"Project folder was found, but dataset ZIP is missing: {SOURCE_ZIP}"
    )

RUNS_DIR = DRIVE_DIR / "runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)

print("Shared project folder:", DRIVE_DIR)
print("Dataset ZIP:", SOURCE_ZIP)
print("Persistent runs:", RUNS_DIR)


# This is the checksum used by the team's 00_shared_dataset_colab.ipynb
# and 01_wideresnet50_2.ipynb for parkinson_dataset_v1.
DATASET_SHA256 = (
    "85efa52f80adbd357537842146ae107e3cd6b9817eb"
    "7ec95beba2644d813f1c6"
)

DATA_ROOT = Path("/content/ParkinsonData")
DATASET_DIR = DATA_ROOT / "datasets"
MANIFEST = DATASET_DIR / "annotations" / "fold_manifest.csv"

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def verify_extracted_dataset():
    metadata_path = DATA_ROOT / "dataset_package.json"
    if not metadata_path.is_file() or not MANIFEST.is_file():
        return None

    try:
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        if metadata.get("dataset_version") != "parkinson_dataset_v1":
            return None

        for relative_name, info in metadata.get("files", {}).items():
            target = (DATA_ROOT / relative_name).resolve()
            if not target.is_relative_to(DATA_ROOT.resolve()):
                return None
            if not target.is_file():
                return None
            if sha256_file(target) != info["sha256"]:
                return None

        return metadata
    except (OSError, ValueError, KeyError, TypeError):
        return None

def extract_checked(source: Path, destination: Path, expected_sha256: str) -> dict:
    if not source.is_file():
        raise FileNotFoundError(f"Dataset ZIP not found: {source}")

    local_zip = Path("/content/parkinson_dataset_v1.zip")
    print("Copying the shared dataset ZIP to Colab local storage...")
    shutil.copy2(source, local_zip)

    actual_sha256 = sha256_file(local_zip)
    print("Expected SHA256:", expected_sha256)
    print("Actual SHA256:  ", actual_sha256)
    if actual_sha256 != expected_sha256:
        raise ValueError(
            "Dataset ZIP checksum mismatch. Extraction stopped; do not bypass this check."
        )

    if destination.exists():
        shutil.rmtree(destination)
    destination.mkdir(parents=True, exist_ok=True)
    destination_root = destination.resolve()

    with zipfile.ZipFile(local_zip, "r") as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(destination_root):
                raise ValueError(f"Unsafe ZIP path: {member.filename}")
            unix_mode = member.external_attr >> 16
            if stat.S_ISLNK(unix_mode):
                raise ValueError(f"Symbolic links are not allowed in dataset ZIP: {member.filename}")
        archive.extractall(destination)

    metadata_path = destination / "dataset_package.json"
    if not metadata_path.is_file():
        raise FileNotFoundError("dataset_package.json is missing after extraction.")
    return json.loads(metadata_path.read_text(encoding="utf-8"))

# Reuse an existing extraction only after checking every packaged file hash.
dataset_metadata = verify_extracted_dataset()
if dataset_metadata is not None:
    print("Existing extracted dataset is intact; reusing it.")
else:
    print("No intact extraction found; extracting the verified ZIP.")
    dataset_metadata = extract_checked(SOURCE_ZIP, DATA_ROOT, DATASET_SHA256)

# Verify the package contents after extraction (or reuse).
if dataset_metadata.get("dataset_version") != "parkinson_dataset_v1":
    raise ValueError("Unexpected dataset version.")
for relative_name, info in dataset_metadata.get("files", {}).items():
    target = (DATA_ROOT / relative_name).resolve()
    if not target.is_relative_to(DATA_ROOT.resolve()):
        raise ValueError(f"Unsafe dataset metadata path: {relative_name}")
    if not target.is_file() or sha256_file(target) != info["sha256"]:
        raise ValueError(f"Dataset file failed SHA256 verification: {relative_name}")

if not MANIFEST.is_file():
    raise FileNotFoundError(f"Fold manifest is missing: {MANIFEST}")

# ---------- Fetch the exact, reproducible DeepRBF source revision ----------
REPOSITORY_URL = "https://github.com/Aihan-2005/ParkinsonAiResearch.git"
CODE_COMMIT = "a1d1bfcb3e0234c24a3c9c0c81ece6ace26ab835"
PROJECT_DIR = Path("/content/ParkinsonAiResearch")

# PROJECT_DIR is a temporary Colab workspace, not the persistent Drive folder.
if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

subprocess.run(
    ["git", "clone", "--quiet", REPOSITORY_URL, str(PROJECT_DIR)],
    check=True,
)
subprocess.run(
    ["git", "-C", str(PROJECT_DIR), "checkout", "--quiet", "--detach", CODE_COMMIT],
    check=True,
)
actual_commit = subprocess.check_output(
    ["git", "-C", str(PROJECT_DIR), "rev-parse", "HEAD"],
    text=True,
).strip()
if actual_commit != CODE_COMMIT:
    raise RuntimeError(
        f"Unexpected Git revision {actual_commit}; expected {CODE_COMMIT}."
    )

required_code = [
    "train.py",
    "train_all_folds.py",
    "train_deep_rbf.py",
    "train_deep_rbf_all_folds.py",
    "tune_deep_rbf_thresholds_all_folds.py",
    "export_deep_rbf_bundle.py",
    "eval_deep_rbf.py",
    "model/deep_rbf_system.py",
]
missing_code = [name for name in required_code if not (PROJECT_DIR / name).is_file()]
if missing_code:
    raise FileNotFoundError(f"GitHub code snapshot is incomplete: {missing_code}")

# Compatibility fix for the pinned DeepRBF-only commit:
# reject_distances() now implements the thesis "both_inside" gap rule directly
# and no longer accepts the retired gap_scope keyword.
def patch_obsolete_gap_scope(relative_path: str) -> None:
    path = PROJECT_DIR / relative_path
    if not path.is_file():
        raise FileNotFoundError(f"Expected source file is missing: {path}")

    original = path.read_text(encoding="utf-8")
    updated = original

    if relative_path == "tests/test_deep_rbf_thesis.py":
        # This final test compares the retired CentreRBF/global-gap behavior
        # with the old optional gap_scope API. CentreRBF is not part of this
        # DeepRBF implementation, so the test is obsolete and must be removed.
        updated, removed_legacy = re.subn(
            r"(?ms)^def test_legacy_centre_rbf_gap_rule_remains_global_by_default\(\):.*\Z",
            "",
            updated,
        )
        if removed_legacy not in (0, 1):
            raise RuntimeError(
                "Unexpected number of obsolete legacy CentreRBF tests: "
                f"{removed_legacy}"
            )

    # The old source places gap_scope on a separate line in some callers
    # and on the same line in tests. Remove only this obsolete keyword.
    # Keep the preceding argument's comma, so multiline calls remain valid.
    gap_scope_pattern = re.compile(
        r"\s*gap_scope\s*=\s*(['\"])both_inside\1\s*,?"
    )
    updated, removed_gap_scope = gap_scope_pattern.subn("", updated)

    if re.search(r"\bgap_scope\s*=", updated):
        raise RuntimeError(
            f"Unsupported gap_scope call remains in {relative_path}; "
            "source was not changed."
        )

    if updated != original:
        # Validate before writing and keep an untouched backup.
        ast.parse(updated, filename=str(path))
        backup = path.with_name(path.name + ".pre_gap_scope_fix")
        if not backup.exists():
            backup.write_text(original, encoding="utf-8")

        temporary = path.with_name(path.name + ".gap_scope_fix.tmp")
        temporary.write_text(updated, encoding="utf-8")
        ast.parse(temporary.read_text(encoding="utf-8"), filename=str(temporary))
        temporary.replace(path)

        print(
            f"Updated {relative_path}: removed {removed_gap_scope} obsolete "
            "gap_scope keyword(s); backup preserved."
        )
    else:
        # Idempotent: the setup cell can be rerun safely.
        print(f"{relative_path}: already compatible; no change needed.")


for relative_path in (
    "utils/rbf_pipeline.py",
    "utils/rbf_thresholds.py",
    "tests/test_deep_rbf_thesis.py",
):
    patch_obsolete_gap_scope(relative_path)

# Ensure no Python source file still calls the removed API.
remaining_gap_scope = []
for source_path in PROJECT_DIR.rglob("*.py"):
    if ".git" in source_path.parts:
        continue
    for line_number, line in enumerate(
        source_path.read_text(encoding="utf-8").splitlines(), start=1
    ):
        if re.search(r"\bgap_scope\s*=", line):
            remaining_gap_scope.append(
                f"{source_path.relative_to(PROJECT_DIR)}:{line_number}: {line.strip()}"
            )

if remaining_gap_scope:
    raise RuntimeError(
        "Obsolete gap_scope references remain:\n"
        + "\n".join(remaining_gap_scope)
    )

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))

# Keep Colab's CUDA-compatible PyTorch/torchvision; install other requirements only.
requirements = [
    line.strip()
    for line in (PROJECT_DIR / "requirements.txt").read_text(encoding="utf-8").splitlines()
    if line.strip()
    and not line.lstrip().startswith("#")
    and not line.lower().startswith(("torch", "torchvision"))
]
if requirements:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *requirements],
        check=True,
    )

import torchvision
import model.rbf as rbf_module
from model.rbf import DeepRBF, reject_distances

if not hasattr(DeepRBF, "forward") or hasattr(rbf_module, "CentreRBF"):
    raise RuntimeError("The checked-out source is not the expected DeepRBF-only implementation.")

# Verify the current rejection API before training.
reject_distances(
    torch.tensor([[0.2,  0.8]], device="cuda"),
    distance_threshold=1.0,
    gap_threshold=0.05,
)

DEVICE = "cuda"
TORCH_DEVICE = torch.device("cuda")
SETUP_COMPLETE = True

print("\n" + "=" * 60)
print("SETUP COMPLETE")
print("=" * 60)
print("Colab GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__, "| torchvision:", torchvision.__version__)
print("Git commit:", actual_commit)
print("Dataset version:", dataset_metadata.get("dataset_version"))
print("Images:", dataset_metadata.get("images"), "| traces:", dataset_metadata.get("traces"))
print("Project:", PROJECT_DIR)
print("Dataset:", DATASET_DIR)
print("Manifest:", MANIFEST)
print("Persistent Drive runs:", RUNS_DIR)
print("Dataset SHA256 and packaged file hashes: VERIFIED")
print("Ready: training will execute on the Colab GPU.")


Mounted at /content/drive
Shared project folder: /content/drive/.shortcut-targets-by-id/1UGKWBfKn2fvGjBr39P1v9ndyA5_8A2Gs/ParkinsonResearch
Dataset ZIP: /content/drive/.shortcut-targets-by-id/1UGKWBfKn2fvGjBr39P1v9ndyA5_8A2Gs/ParkinsonResearch/datasets/parkinson_dataset_v1.zip
Persistent runs: /content/drive/.shortcut-targets-by-id/1UGKWBfKn2fvGjBr39P1v9ndyA5_8A2Gs/ParkinsonResearch/runs
No intact extraction found; extracting the verified ZIP.
Copying the shared dataset ZIP to Colab local storage...
Expected SHA256: 85efa52f80adbd357537842146ae107e3cd6b9817eb7ec95beba2644d813f1c6
Actual SHA256:   85efa52f80adbd357537842146ae107e3cd6b9817eb7ec95beba2644d813f1c6
Updated utils/rbf_pipeline.py: removed 1 obsolete gap_scope keyword(s); backup preserved.
Updated utils/rbf_thresholds.py: removed 1 obsolete gap_scope keyword(s); backup preserved.
Updated tests/test_deep_rbf_thesis.py: removed 2 obsolete gap_scope keyword(s); backup preserved.

SETUP COMPLETE
Colab GPU: Tesla T4
PyTorch: 2.14.1

In [2]:
patcher_source = '#!/usr/bin/env python3\n"""Patch train.py to write compact, non-resumable checkpoints after final epochs.\n\nIntermediate checkpoints remain fully resumable. Only a checkpoint written at\nits configured final epoch drops optimizer/RNG state. Model weights and all\ninference/provenance metadata are preserved exactly.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport ast\nfrom pathlib import Path\n\nOLD_RESUME_GUARD = \'\'\'        if "optimizer_state_dict" not in resume_checkpoint:\n            raise SystemExit("This checkpoint has no optimizer state and cannot be resumed.")\n\'\'\'\nNEW_RESUME_GUARD = \'\'\'        has_optimizer_state = "optimizer_state_dict" in resume_checkpoint\n        compact_complete = (\n            resume_checkpoint.get("checkpoint_storage") == "compact_completed"\n            and resume_checkpoint.get("training_complete") is True\n            and int(resume_checkpoint.get("epoch", -1)) == args.epochs\n        )\n        if not has_optimizer_state and not compact_complete:\n            raise SystemExit(\n                "This checkpoint has no optimizer state. Only a compact checkpoint "\n                "from a fully completed run with the same epoch budget can be reused. "\n                "An incomplete compact checkpoint cannot be resumed."\n            )\n\'\'\'\n\nOLD_OPTIMIZER_LOAD = \'\'\'        optimizer.load_state_dict(resume_checkpoint["optimizer_state_dict"])\n\'\'\'\nNEW_OPTIMIZER_LOAD = \'\'\'        if "optimizer_state_dict" in resume_checkpoint:\n            optimizer.load_state_dict(resume_checkpoint["optimizer_state_dict"])\n\'\'\'\n\nOLD_FINAL_SAVE = \'\'\'                _save_checkpoint(last_path, checkpoint_payload(epoch, val_stats))\n\'\'\'\nNEW_FINAL_SAVE = \'\'\'                payload = checkpoint_payload(epoch, val_stats)\n                if epoch == args.epochs:\n                    # A completed fold is used for frozen feature extraction and\n                    # inference. Its optimizer and RNG states are no longer needed.\n                    # Keep every model tensor and all provenance/training metadata.\n                    for key in (\n                        "optimizer_state_dict",\n                        "train_loader_generator_state",\n                        "torch_rng_state",\n                        "cuda_rng_state_all",\n                    ):\n                        payload.pop(key, None)\n                    payload["checkpoint_storage"] = "compact_completed"\n                    payload["training_complete"] = True\n                else:\n                    payload["checkpoint_storage"] = "resumable_full"\n                    payload["training_complete"] = False\n                _save_checkpoint(last_path, payload)\n\'\'\'\n\n\ndef _replace_once(source: str, old: str, new: str, label: str) -> str:\n    # Check the complete replacement first: an old indented line can appear as\n    # a substring inside a newly inserted, more-indented block.\n    if new in source:\n        return source  # already patched\n    count = source.count(old)\n    if count == 1:\n        return source.replace(old, new, 1)\n    raise RuntimeError(\n        f"Could not safely patch {label}: expected one original block or an already-patched block; "\n        f"found {count} original matches. No file was changed."\n    )\n\n\ndef patch_train_file(train_file: Path) -> bool:\n    train_file = Path(train_file)\n    if not train_file.is_file():\n        raise FileNotFoundError(f"train.py not found: {train_file}")\n\n    original = train_file.read_text(encoding="utf-8")\n    updated = original\n    updated = _replace_once(updated, OLD_RESUME_GUARD, NEW_RESUME_GUARD, "compact resume guard")\n    updated = _replace_once(updated, OLD_OPTIMIZER_LOAD, NEW_OPTIMIZER_LOAD, "optimizer restore")\n    updated = _replace_once(updated, OLD_FINAL_SAVE, NEW_FINAL_SAVE, "final checkpoint save")\n\n    if updated == original:\n        print("Compact checkpoint support is already installed in train.py.")\n        return False\n\n    ast.parse(updated, filename=str(train_file))\n    backup = train_file.with_name("train.py.pre_compact_backup")\n    if not backup.exists():\n        backup.write_text(original, encoding="utf-8")\n\n    temporary = train_file.with_name("train.py.compact.tmp")\n    temporary.write_text(updated, encoding="utf-8")\n    ast.parse(temporary.read_text(encoding="utf-8"), filename=str(temporary))\n    temporary.replace(train_file)\n\n    # Confirm all patch markers and ensure the new code remains syntactically valid.\n    final_source = train_file.read_text(encoding="utf-8")\n    ast.parse(final_source, filename=str(train_file))\n    required_markers = (\n        \'"checkpoint_storage"] = "compact_completed"\',\n        \'"training_complete"] = True\',\n        \'if "optimizer_state_dict" in resume_checkpoint:\',\n    )\n    missing = [marker for marker in required_markers if marker not in final_source]\n    if missing:\n        raise RuntimeError(f"Patch verification failed; missing markers: {missing}")\n\n    print(f"Patched: {train_file}")\n    print(f"Backup:  {backup}")\n    print("Intermediate checkpoints: full/resumable")\n    print("Final completed checkpoints: compact (optimizer and RNG states removed)")\n    print("Model weights, loss metadata, preprocessing, fold identity, manifest hash and history are preserved.")\n    return True\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--repo-root", type=Path, required=True)\n    args = parser.parse_args()\n    patch_train_file(args.repo_root / "train.py")\n\n\nif __name__ == "__main__":\n    main()\n'
compactor_source = '#!/usr/bin/env python3\n"""Safely remove optimizer/RNG tensors from a completed Focal WideResNet checkpoint.\n\nThe model state_dict and all provenance/training metadata are preserved. The\nscript refuses to alter a baseline if a DeepRBF checkpoint already references\nits SHA-256, because changing the baseline file would invalidate that link.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport os\nimport shutil\nfrom pathlib import Path\n\nimport torch\n\nDROP_KEYS = {\n    "optimizer_state_dict",\n    "train_loader_generator_state",\n    "torch_rng_state",\n    "cuda_rng_state_all",\n}\n\n\ndef sha256_file(path: Path) -> str:\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef tensor_bytes(value) -> int:\n    if isinstance(value, torch.Tensor):\n        return value.numel() * value.element_size()\n    if isinstance(value, dict):\n        return sum(tensor_bytes(item) for item in value.values())\n    if isinstance(value, (tuple, list)):\n        return sum(tensor_bytes(item) for item in value)\n    return 0\n\n\ndef _same_state_dict(left: dict, right: dict) -> bool:\n    if left.keys() != right.keys():\n        return False\n    for key in left:\n        a, b = left[key], right[key]\n        if isinstance(a, torch.Tensor):\n            if not isinstance(b, torch.Tensor) or not torch.equal(a, b):\n                return False\n        elif a != b:\n            return False\n    return True\n\n\ndef compact_checkpoint(\n    checkpoint_path: Path,\n    *,\n    fold_id: int,\n    expected_epochs: int,\n    deep_rbf_root: Path | None = None,\n) -> bool:\n    checkpoint_path = Path(checkpoint_path)\n    if not checkpoint_path.is_file():\n        raise FileNotFoundError(checkpoint_path)\n\n    original_hash = sha256_file(checkpoint_path)\n    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)\n    if not isinstance(checkpoint, dict):\n        raise ValueError("Expected a dictionary checkpoint")\n    if checkpoint.get("checkpoint_kind") != "last":\n        raise ValueError("Only last.pt checkpoints can be compacted")\n    if checkpoint.get("architecture") != "wide_resnet50_2":\n        raise ValueError("Expected a WideResNet50-2 checkpoint")\n    if checkpoint.get("label_source") != "consensus":\n        raise ValueError("Expected the consensus-label baseline")\n    if checkpoint.get("loss", {}).get("name") != "focal":\n        raise ValueError("Expected a Focal Loss baseline")\n    if int(checkpoint.get("fold_id", -1)) != fold_id:\n        raise ValueError("Checkpoint fold_id does not match --fold-id")\n    if int(checkpoint.get("epoch", -1)) != expected_epochs:\n        raise ValueError(\n            f"Checkpoint is not complete: epoch={checkpoint.get(\'epoch\')}, "\n            f"expected={expected_epochs}"\n        )\n    if "model_state_dict" not in checkpoint:\n        raise ValueError("Checkpoint is missing model_state_dict")\n\n    if checkpoint.get("checkpoint_storage") == "compact_completed":\n        print(f"Already compact: {checkpoint_path}")\n        return False\n\n    if deep_rbf_root is not None:\n        # A DeepRBF stage checkpoint stores the exact baseline file hash.\n        # Never change a baseline that has already been used to train DeepRBF.\n        candidates = list(Path(deep_rbf_root).glob(f"fold{fold_id:02d}_*/last.pt"))\n        if candidates:\n            raise RuntimeError(\n                f"Refusing to compact {checkpoint_path}: a DeepRBF checkpoint already exists "\n                f"for fold {fold_id} ({candidates[0]}). The DeepRBF source-baseline SHA-256 "\n                "may depend on the current checkpoint."\n            )\n\n    if "optimizer_state_dict" not in checkpoint:\n        raise ValueError("Checkpoint is neither a full resumable checkpoint nor a recognized compact checkpoint")\n\n    original_model_state = checkpoint["model_state_dict"]\n    original_model_bytes = tensor_bytes(original_model_state)\n    compact = {key: value for key, value in checkpoint.items() if key not in DROP_KEYS}\n    compact["checkpoint_storage"] = "compact_completed"\n    compact["training_complete"] = True\n    compact["compact_removed_keys"] = sorted(DROP_KEYS.intersection(checkpoint))\n    compact["source_full_checkpoint_sha256"] = original_hash\n\n    needed = int(tensor_bytes(compact) * 1.05) + 32 * 1024**2\n    free = shutil.disk_usage(checkpoint_path.parent).free\n    if free < needed:\n        raise RuntimeError(\n            f"Not enough temporary space to compact safely: need {needed / 1e9:.2f} GB, "\n            f"free {free / 1e9:.2f} GB. Original checkpoint was not changed."\n        )\n\n    temporary = checkpoint_path.with_name(checkpoint_path.name + ".compact.tmp")\n    try:\n        torch.save(compact, temporary)\n        verified = torch.load(temporary, map_location="cpu", weights_only=True)\n        if not _same_state_dict(original_model_state, verified.get("model_state_dict", {})):\n            raise RuntimeError("Model weights changed during compact checkpoint serialization")\n        if any(key in verified for key in DROP_KEYS):\n            raise RuntimeError("Optimizer/RNG state remains in the compact checkpoint")\n        for key in (\n            "architecture", "fold_id", "epoch", "loss", "training_config",\n            "fold_manifest_sha256", "preprocessing", "class_names", "num_classes",\n        ):\n            if verified.get(key) != checkpoint.get(key):\n                raise RuntimeError(f"Checkpoint metadata changed during compaction: {key}")\n        if verified.get("checkpoint_storage") != "compact_completed":\n            raise RuntimeError("Compact checkpoint marker is missing")\n        if sha256_file(checkpoint_path) != original_hash:\n            raise RuntimeError("Original checkpoint changed before atomic replacement")\n        os.replace(temporary, checkpoint_path)\n    finally:\n        if temporary.exists():\n            temporary.unlink()\n\n    final = torch.load(checkpoint_path, map_location="cpu", weights_only=True)\n    if not _same_state_dict(original_model_state, final["model_state_dict"]):\n        raise RuntimeError("Post-replacement model-weight verification failed")\n\n    new_size = checkpoint_path.stat().st_size\n    print(f"Compacted: {checkpoint_path}")\n    print(f"Model weights preserved: {original_model_bytes / 1024**2:.2f} MB")\n    print(f"New checkpoint size: {new_size / 1024**2:.2f} MB")\n    print(f"Original SHA-256: {original_hash}")\n    print(f"Compact SHA-256:  {sha256_file(checkpoint_path)}")\n    return True\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--checkpoint", type=Path, required=True)\n    parser.add_argument("--fold-id", type=int, required=True)\n    parser.add_argument("--expected-epochs", type=int, default=15)\n    parser.add_argument("--deep-rbf-root", type=Path, default=None)\n    args = parser.parse_args()\n    compact_checkpoint(\n        args.checkpoint,\n        fold_id=args.fold_id,\n        expected_epochs=args.expected_epochs,\n        deep_rbf_root=args.deep_rbf_root,\n    )\n\n\nif __name__ == "__main__":\n    main()\n'

patcher_path = PROJECT_DIR / "apply_compact_checkpoint_patch.py"
compactor_path = PROJECT_DIR / "compact_wideresnet_checkpoint.py"
patcher_path.write_text(patcher_source, encoding="utf-8")
compactor_path.write_text(compactor_source, encoding="utf-8")
subprocess.run(
    [sys.executable, str(patcher_path), "--repo-root", str(PROJECT_DIR)],
    cwd=PROJECT_DIR,
    check=True,
)
print("Compact completed-checkpoint support is active.")


Compact completed-checkpoint support is active.


## 0A. Preflight — verify the DeepRBF implementation before spending GPU time

This runs the focused DeepRBF architecture, loss, rejection-rule, and combined-bundle tests. Training cells should only be run if this cell passes.

The setup also installs a compact-checkpoint patch in `train.py`. Intermediate checkpoints remain full and resumable; at the final epoch, the exact model weights and provenance metadata are retained while optimizer and RNG states are removed. The already-trained Fold 1 checkpoint is not rewritten because its existing DeepRBF checkpoint is bound to its original SHA-256.

The setup removes the obsolete `gap_scope` keyword from the old callers and removes one legacy CentreRBF-only test. The current DeepRBF `reject_distances()` already implements the thesis `both_inside` rule directly; the correction does not change that rule or the model architecture.


In [3]:
if not globals().get("SETUP_COMPLETE", False):
    raise RuntimeError(
        "Setup did not complete successfully. Fix the Setup cell first; "
        "do not run preflight or training after a Setup error."
    )

smoke_test_command = [
    sys.executable, "-m", "pytest", "-q",
    "tests/test_deep_rbf_thesis.py",
    "tests/test_deep_rbf_system.py",
]
subprocess.run(smoke_test_command, cwd=PROJECT_DIR, check=True)
print("DeepRBF preflight tests passed. GPU:", torch.cuda.get_device_name(0))

DeepRBF preflight tests passed. GPU: Tesla T4


## 1. Configuration — full 25-fold development run

Fold 1 already exists and will be reused. Folds 2–25 will be trained. The measured Fold 1 checkpoint contains about **255.24 MB of model weights** and **478.47 MB of optimizer state**. The compact policy keeps the exact model weights and provenance metadata, while removing optimizer/RNG state only after the final epoch.

The notebook calculates the storage requirement from the existing checkpoint(s), existing DeepRBF files, and any completed bundles. With the observed Fold 1 sizes, the additional requirement for folds 2–25 is expected to be around **8–9 GB**, including a temporary full-checkpoint allowance, one bundle staged at a time, and a safety margin. This is an estimate, not a Google Drive quota guarantee.

Set `DRIVE_FREE_GB` below to the **actual free storage** currently available in the Google account. Do not enter the account's total 15 GB quota unless all 15 GB are genuinely free.

Keep `DELETE_STAGE_CHECKPOINTS_AFTER_BUNDLE = True` for a 15 GB account. Once a combined bundle has been exported, hash-checked, loaded, and validation-evaluated, the duplicate source `.pt` files are removed. The combined bundle, training histories, threshold decisions, and validation reports remain in Drive.

Do not change the training recipe halfway through a run. If you intentionally change it, use a new output directory.


In [6]:
# Full development run: Fold 1 is reused; missing folds are trained.
FOLD_START = 1
FOLD_END = 25

SEED = 42

# Stage 1: WideResNet50-2 + Focal Loss (same recipe as the team's baseline notebook)
BASELINE_EPOCHS = 15
BASELINE_BATCH_SIZE = 8
HEAD_LR = 1e-3
BACKBONE_LR = 1e-5
WEIGHT_DECAY = 1e-4
FOCAL_GAMMA = 2.0
FOCAL_ALPHA_MODE = "uniform"
AUGMENTATION = "standard"
SAMPLING = "class_balanced"
ROTATION_DEGREES = 10.0
HFLIP_PROB = 0.5
NOISE_STD = 0.01
CHECKPOINT_EVERY = 5

# Stage 2: DeepRBF
RBF_EPOCHS = 30
RBF_DIM = 64
RBF_DISTANCE = "l1"       # L1 and L2 are separate experiments.
RBF_DISTANCE_POWER = "norm"
KNOWN_MARGIN = 1.0
REJECT_MARGIN = 1.0
BETA = 1.0
RBF_LR = 1e-3
RBF_WEIGHT_DECAY = 0.0
FEATURE_BATCH_SIZE = 8

# Remove duplicate stage checkpoints only after bundle + validation verification.
DELETE_STAGE_CHECKPOINTS_AFTER_BUNDLE = True

# Enter actual free space from Google Drive storage settings (in decimal GB).
# Example: 12.4 means 12.4 GB free now, not the account's total quota.
DRIVE_FREE_GB = 13.0

BASELINE_RUNS = DRIVE_DIR / "runs" / "baselines_shared_dataset_v1"
BASELINE_DIR = BASELINE_RUNS / "focal" / "checkpoints"
RBF_ROOT = DRIVE_DIR / "runs" / "deeprbf_focal_shared_dataset_v1"

BASELINE_DIR.mkdir(parents=True, exist_ok=True)
RBF_ROOT.mkdir(parents=True, exist_ok=True)

selected_fold_ids = list(range(FOLD_START, FOLD_END + 1))
if not (1 <= FOLD_START <= FOLD_END <= 25):
    raise ValueError("Fold range must satisfy 1 <= FOLD_START <= FOLD_END <= 25.")

if DRIVE_FREE_GB is None:
    raise RuntimeError(
        "Set DRIVE_FREE_GB to the actual free Google Drive storage before training."
    )
if DRIVE_FREE_GB <= 0:
    raise ValueError("DRIVE_FREE_GB must be positive.")

def tensor_bytes(value):
    if isinstance(value, torch.Tensor):
        return value.numel() * value.element_size()
    if isinstance(value, dict):
        return sum(tensor_bytes(v) for v in value.values())
    if isinstance(value, (list, tuple)):
        return sum(tensor_bytes(v) for v in value)
    return 0

# Discover existing WideResNet checkpoints.
existing_baselines = {}
model_state_bytes = None
for path in sorted(BASELINE_DIR.glob("*_last.pt")):
    payload = torch.load(path, map_location="cpu", weights_only=True)
    fold_id = int(payload.get("fold_id", -1))
    if fold_id not in selected_fold_ids:
        del payload
        continue
    if fold_id in existing_baselines:
        raise RuntimeError(f"Multiple WideResNet checkpoints found for fold {fold_id}.")
    if payload.get("architecture") != "wide_resnet50_2":
        raise RuntimeError(f"Unexpected architecture in {path}.")
    if payload.get("loss", {}).get("name") != "focal":
        raise RuntimeError(f"Fold {fold_id} checkpoint is not a Focal Loss baseline.")
    if payload.get("fold_manifest_sha256") != sha256_file(MANIFEST):
        raise RuntimeError(f"Fold {fold_id} checkpoint uses a different fold manifest.")
    if int(payload.get("epoch", -1)) > BASELINE_EPOCHS:
        raise RuntimeError(f"Fold {fold_id} checkpoint exceeds the configured epoch budget.")
    existing_baselines[fold_id] = path
    if model_state_bytes is None:
        model_state_bytes = tensor_bytes(payload["model_state_dict"])
    del payload

# Discover already exported combined bundles. A completed bundle replaces its
# two source .pt files for inference and must not be counted as missing training.
existing_bundles = {}
bundle_model_bytes = None
bundle_rbf_bytes = None
for fold_id in selected_fold_ids:
    bundle_path = (
        RBF_ROOT / f"fold{fold_id:02d}_{RBF_DISTANCE}_{RBF_DISTANCE_POWER}"
        / "combined_model.pt"
    )
    if bundle_path.is_file():
        bundle = torch.load(bundle_path, map_location="cpu", weights_only=True)
        if int(bundle.get("fold_id", -1)) != fold_id:
            raise RuntimeError(f"Combined bundle has the wrong fold ID: {bundle_path}")
        existing_bundles[fold_id] = bundle_path
        if bundle_model_bytes is None:
            bundle_model_bytes = tensor_bytes(bundle["baseline"]["model_state_dict"])
            bundle_rbf_bytes = tensor_bytes(bundle["deep_rbf"]["model_state_dict"])
        del bundle

# Discover existing DeepRBF training checkpoints.
existing_rbf = {}
rbf_state_bytes = bundle_rbf_bytes
for path in sorted(RBF_ROOT.glob(f"fold*_{RBF_DISTANCE}_{RBF_DISTANCE_POWER}/last.pt")):
    match = re.match(r"fold(\d{2})_", path.parent.name)
    if not match:
        continue
    fold_id = int(match.group(1))
    if fold_id not in selected_fold_ids:
        continue
    if fold_id in existing_rbf:
        raise RuntimeError(f"Multiple DeepRBF checkpoints found for fold {fold_id}.")
    existing_rbf[fold_id] = path
    if rbf_state_bytes is None:
        payload = torch.load(path, map_location="cpu", weights_only=True)
        rbf_state_bytes = tensor_bytes(payload.get("model_state_dict", {}))
        del payload

# A DeepRBF checkpoint without its source baseline or a combined bundle is unsafe:
# the RBF checkpoint is cryptographically tied to that exact baseline file.
for fold_id in existing_rbf:
    if fold_id not in existing_baselines and fold_id not in existing_bundles:
        raise RuntimeError(
            f"Fold {fold_id} has a DeepRBF checkpoint but no matching baseline checkpoint "
            "or combined bundle. Do not continue until its source baseline is restored."
        )

if model_state_bytes is None:
    model_state_bytes = bundle_model_bytes
if model_state_bytes is None:
    raise RuntimeError(
        "Cannot estimate model storage: no existing WideResNet checkpoint or combined bundle "
        "was found. Keep the completed Fold 1 baseline in Drive."
    )
if rbf_state_bytes is None:
    rbf_state_bytes = 5 * 1024**2  # conservative first-fold estimate

missing_baseline_folds = [
    f for f in selected_fold_ids
    if f not in existing_baselines and f not in existing_bundles
]
missing_rbf_folds = [
    f for f in selected_fold_ids
    if f not in existing_rbf and f not in existing_bundles
]
missing_bundle_folds = [
    f for f in selected_fold_ids if f not in existing_bundles
]

additional_baseline_bytes = len(missing_baseline_folds) * model_state_bytes
additional_rbf_bytes = len(missing_rbf_folds) * max(rbf_state_bytes, 5 * 1024**2)

# Only one combined bundle is staged at a time when source-checkpoint cleanup is on.
bundle_bytes = model_state_bytes + max(rbf_state_bytes, 5 * 1024**2) + 5 * 1024**2
if DELETE_STAGE_CHECKPOINTS_AFTER_BUNDLE:
    bundle_reserve = bundle_bytes if missing_bundle_folds else 0
else:
    bundle_reserve = bundle_bytes * len(missing_bundle_folds)

# Full Adam checkpoint (~734 MB measured) can coexist with its temporary save.
full_checkpoint_temp_reserve = 800_000_000
safety_margin = 1_000_000_000

estimated_required_bytes = (
    additional_baseline_bytes
    + additional_rbf_bytes
    + bundle_reserve
    + full_checkpoint_temp_reserve
    + safety_margin
)
estimated_required_gb = estimated_required_bytes / 1e9

if DRIVE_FREE_GB < estimated_required_gb:
    raise RuntimeError(
        f"Insufficient Drive space: {DRIVE_FREE_GB:.2f} GB free; "
        f"estimated additional requirement is {estimated_required_gb:.2f} GB. "
        "No training was started. Free space or use a larger storage plan."
    )

print("Folds:", FOLD_START, "to", FOLD_END)
print("GPU:", torch.cuda.get_device_name(0))
print("Existing WideResNet folds:", sorted(existing_baselines))
print("Existing DeepRBF folds:", sorted(existing_rbf))
print("Existing combined bundles:", sorted(existing_bundles))
print(f"Model state size per fold: {model_state_bytes / 1e6:.2f} MB")
print(f"Estimated additional compact baselines: {additional_baseline_bytes / 1e9:.2f} GB")
print(f"Estimated additional DeepRBF checkpoints: {additional_rbf_bytes / 1e9:.2f} GB")
print(f"Bundle staging reserve: {bundle_reserve / 1e9:.2f} GB")
print(f"Temporary checkpoint + safety reserve: {(full_checkpoint_temp_reserve + safety_margin) / 1e9:.2f} GB")
print(f"Estimated additional Drive requirement: {estimated_required_gb:.2f} GB")
print("Focal checkpoints:", BASELINE_DIR)
print("DeepRBF outputs:", RBF_ROOT)


Folds: 1 to 25
GPU: Tesla T4
Existing WideResNet folds: [1]
Existing DeepRBF folds: [1]
Existing combined bundles: []
Model state size per fold: 267.63 MB
Estimated additional compact baselines: 6.42 GB
Estimated additional DeepRBF checkpoints: 0.13 GB
Bundle staging reserve: 0.28 GB
Temporary checkpoint + safety reserve: 1.80 GB
Estimated additional Drive requirement: 8.63 GB
Focal checkpoints: /content/drive/.shortcut-targets-by-id/1UGKWBfKn2fvGjBr39P1v9ndyA5_8A2Gs/ParkinsonResearch/runs/baselines_shared_dataset_v1/focal/checkpoints
DeepRBF outputs: /content/drive/.shortcut-targets-by-id/1UGKWBfKn2fvGjBr39P1v9ndyA5_8A2Gs/ParkinsonResearch/runs/deeprbf_focal_shared_dataset_v1


## 2. Stage 1 — train WideResNet50-2 with Focal Loss

This uses the same `train_all_folds.py` training script, fold manifest, augmentation settings, optimizer settings, checkpoint cadence, and Drive checkpoint layout as the teammate's `01_wideresnet50_2.ipynb`.

Only the **Focal Loss** experiment is run here; the CE comparison is not needed for the requested DeepRBF pipeline. `--resume` reuses completed matching folds and resumes incomplete ones.

**Storage policy:** intermediate epoch checkpoints remain full so interrupted training can resume. At the final epoch, the checkpoint keeps the full WideResNet weights, Focal metadata, fold/manifest/preprocessing identity and history, but drops the optimizer and RNG states. Fold 1 is reused unchanged because its existing DeepRBF checkpoint is cryptographically bound to that original baseline file.


In [7]:
from utils.notebook_training import run_with_progress

for fold in range(FOLD_START, FOLD_END + 1):
    fold_name = f"fold{fold:02d}_{RBF_DISTANCE}_{RBF_DISTANCE_POWER}"
    bundle_path = RBF_ROOT / fold_name / "combined_model.pt"
    if bundle_path.is_file():
        print(f"Fold {fold:02d}: verified combined bundle already exists; skip Stage 1.")
        continue

    baseline_command = [
        sys.executable, "-u", "train_all_folds.py",
        "--fold-start", str(fold), "--fold-end", str(fold),
        "--loss", "focal", "--focal-gamma", str(FOCAL_GAMMA),
        "--focal-alpha-mode", FOCAL_ALPHA_MODE,
        "--augmentation", AUGMENTATION,
        "--rotation-degrees", str(ROTATION_DEGREES),
        "--hflip-prob", str(HFLIP_PROB), "--noise-std", str(NOISE_STD),
        "--sampling", SAMPLING, "--label-source", "consensus",
        "--pretrained", "--freeze-policy", "early",
        "--epochs", str(BASELINE_EPOCHS), "--batch-size", str(BASELINE_BATCH_SIZE),
        "--image-size", "224", "--num-workers", "0",
        "--lr", str(HEAD_LR), "--backbone-lr", str(BACKBONE_LR),
        "--weight-decay", str(WEIGHT_DECAY), "--seed", str(SEED),
        "--device", DEVICE, "--manifest", str(MANIFEST),
        "--root", str(DATA_ROOT), "--checkpoint-dir", str(BASELINE_DIR),
        "--checkpoint-every", str(CHECKPOINT_EVERY), "--progress-json", "--resume",
    ]
    print("\nStarting/reusing Focal WideResNet fold", fold)
    print(" ".join(baseline_command))
    run_with_progress(
        baseline_command, title="WideResNet50-2 + Focal Loss",
        fold_start=fold, fold_end=fold, epochs=BASELINE_EPOCHS,
    )

    matches = [
        p for p in BASELINE_DIR.glob("*_last.pt")
        if re.search(rf"_fold{fold:02d}_", p.name)
    ]
    if len(matches) != 1:
        raise RuntimeError(f"Expected exactly one final baseline checkpoint for fold {fold}; found {matches}")

    rbf_checkpoint_path = RBF_ROOT / f"fold{fold:02d}_{RBF_DISTANCE}_{RBF_DISTANCE_POWER}" / "last.pt"
    if fold != 1 and not rbf_checkpoint_path.is_file():
        compact_command = [
            sys.executable, "compact_wideresnet_checkpoint.py",
            "--checkpoint", str(matches[0]),
            "--fold-id", str(fold),
            "--expected-epochs", str(BASELINE_EPOCHS),
            "--deep-rbf-root", str(RBF_ROOT),
        ]
        subprocess.run(compact_command, cwd=PROJECT_DIR, check=True)
    elif rbf_checkpoint_path.is_file():
        print(f"Fold {fold:02d}: preserving baseline file because a DeepRBF checkpoint already exists.")

    saved = torch.load(matches[0], map_location="cpu", weights_only=True)
    if int(saved.get("epoch", -1)) != BASELINE_EPOCHS:
        raise RuntimeError(f"Fold {fold} did not reach {BASELINE_EPOCHS} epochs.")
    if fold != 1 and not rbf_checkpoint_path.is_file() and saved.get("checkpoint_storage") != "compact_completed":
        raise RuntimeError(
            f"Fold {fold} is not compact. Refusing to continue; check the compact checkpoint patch."
        )
    print(
        f"Fold {fold:02d} complete | epoch={saved['epoch']} | "
        f"file_size={matches[0].stat().st_size / 1024**2:.1f} MB | "
        f"storage={saved.get('checkpoint_storage', 'legacy_full')}"
    )
    del saved

print("Stage 1 complete for selected folds.")



Starting/reusing Focal WideResNet fold 1
/usr/bin/python3 -u train_all_folds.py --fold-start 1 --fold-end 1 --loss focal --focal-gamma 2.0 --focal-alpha-mode uniform --augmentation standard --rotation-degrees 10.0 --hflip-prob 0.5 --noise-std 0.01 --sampling class_balanced --label-source consensus --pretrained --freeze-policy early --epochs 15 --batch-size 8 --image-size 224 --num-workers 0 --lr 0.001 --backbone-lr 1e-05 --weight-decay 0.0001 --seed 42 --device cuda --manifest /content/ParkinsonData/datasets/annotations/fold_manifest.csv --root /content/ParkinsonData --checkpoint-dir /content/drive/.shortcut-targets-by-id/1UGKWBfKn2fvGjBr39P1v9ndyA5_8A2Gs/ParkinsonResearch/runs/baselines_shared_dataset_v1/focal/checkpoints --checkpoint-every 5 --progress-json --resume



TLO fold 01/25 | loss=focal | freeze=early | seed=42
Command: /usr/bin/python3 -u /content/ParkinsonAiResearch/train.py --fold-id 1 --architecture wide_resnet50_2 --label-source consensus --loss focal --focal-gamma 2.0 --focal-alpha-mode uniform --augmentation standard --rotation-degrees 10.0 --hflip-prob 0.5 --noise-std 0.01 --sampling class_balanced --freeze-policy early --epochs 15 --batch-size 8 --image-size 224 --num-workers 0 --lr 0.001 --backbone-lr 1e-05 --weight-decay 0.0001 --seed 42 --device cuda --manifest /content/ParkinsonData/datasets/annotations/fold_manifest.csv --root /content/ParkinsonData --checkpoint-dir /content/drive/.shortcut-targets-by-id/1UGKWBfKn2fvGjBr39P1v9ndyA5_8A2Gs/ParkinsonResearch/runs/baselines_shared_dataset_v1/focal/checkpoints --checkpoint-every 5 --progress --progress-json --pretrained --resume
Final-epoch validation predictions: /content/drive/.shortcut-targets-by-id/1UGKWBfKn2fvGjBr39P1v9ndyA5_8A2Gs/ParkinsonResearch/runs/baselines_shared_datas


TLO fold 02/25 | loss=focal | freeze=early | seed=42
Command: /usr/bin/python3 -u /content/ParkinsonAiResearch/train.py --fold-id 2 --architecture wide_resnet50_2 --label-source consensus --loss focal --focal-gamma 2.0 --focal-alpha-mode uniform --augmentation standard --rotation-degrees 10.0 --hflip-prob 0.5 --noise-std 0.01 --sampling class_balanced --freeze-policy early --epochs 15 --batch-size 8 --image-size 224 --num-workers 0 --lr 0.001 --backbone-lr 1e-05 --weight-decay 0.0001 --seed 42 --device cuda --manifest /content/ParkinsonData/datasets/annotations/fold_manifest.csv --root /content/ParkinsonData --checkpoint-dir /content/drive/.shortcut-targets-by-id/1UGKWBfKn2fvGjBr39P1v9ndyA5_8A2Gs/ParkinsonResearch/runs/baselines_shared_dataset_v1/focal/checkpoints --checkpoint-every 5 --progress --progress-json --pretrained --resume
Downloading: "https://download.pytorch.org/models/wide_resnet50_2-9ba9bcbe.pth" to /root/.cache/torch/hub/checkpoints/wide_resnet50_2-9ba9bcbe.pth

  0%| 


TLO fold 03/25 | loss=focal | freeze=early | seed=42
Command: /usr/bin/python3 -u /content/ParkinsonAiResearch/train.py --fold-id 3 --architecture wide_resnet50_2 --label-source consensus --loss focal --focal-gamma 2.0 --focal-alpha-mode uniform --augmentation standard --rotation-degrees 10.0 --hflip-prob 0.5 --noise-std 0.01 --sampling class_balanced --freeze-policy early --epochs 15 --batch-size 8 --image-size 224 --num-workers 0 --lr 0.001 --backbone-lr 1e-05 --weight-decay 0.0001 --seed 42 --device cuda --manifest /content/ParkinsonData/datasets/annotations/fold_manifest.csv --root /content/ParkinsonData --checkpoint-dir /content/drive/.shortcut-targets-by-id/1UGKWBfKn2fvGjBr39P1v9ndyA5_8A2Gs/ParkinsonResearch/runs/baselines_shared_dataset_v1/focal/checkpoints --checkpoint-every 5 --progress --progress-json --pretrained --resume
Device=cuda | fold=3/25 | architecture=wide_resnet50_2 | freeze_policy=early | pretrained=True
Label source=consensus | classes=('control', 'parkinson', '


TLO fold 04/25 | loss=focal | freeze=early | seed=42
Command: /usr/bin/python3 -u /content/ParkinsonAiResearch/train.py --fold-id 4 --architecture wide_resnet50_2 --label-source consensus --loss focal --focal-gamma 2.0 --focal-alpha-mode uniform --augmentation standard --rotation-degrees 10.0 --hflip-prob 0.5 --noise-std 0.01 --sampling class_balanced --freeze-policy early --epochs 15 --batch-size 8 --image-size 224 --num-workers 0 --lr 0.001 --backbone-lr 1e-05 --weight-decay 0.0001 --seed 42 --device cuda --manifest /content/ParkinsonData/datasets/annotations/fold_manifest.csv --root /content/ParkinsonData --checkpoint-dir /content/drive/.shortcut-targets-by-id/1UGKWBfKn2fvGjBr39P1v9ndyA5_8A2Gs/ParkinsonResearch/runs/baselines_shared_dataset_v1/focal/checkpoints --checkpoint-every 5 --progress --progress-json --pretrained --resume
Device=cuda | fold=4/25 | architecture=wide_resnet50_2 | freeze_policy=early | pretrained=True
Label source=consensus | classes=('control', 'parkinson', '


TLO fold 05/25 | loss=focal | freeze=early | seed=42
Command: /usr/bin/python3 -u /content/ParkinsonAiResearch/train.py --fold-id 5 --architecture wide_resnet50_2 --label-source consensus --loss focal --focal-gamma 2.0 --focal-alpha-mode uniform --augmentation standard --rotation-degrees 10.0 --hflip-prob 0.5 --noise-std 0.01 --sampling class_balanced --freeze-policy early --epochs 15 --batch-size 8 --image-size 224 --num-workers 0 --lr 0.001 --backbone-lr 1e-05 --weight-decay 0.0001 --seed 42 --device cuda --manifest /content/ParkinsonData/datasets/annotations/fold_manifest.csv --root /content/ParkinsonData --checkpoint-dir /content/drive/.shortcut-targets-by-id/1UGKWBfKn2fvGjBr39P1v9ndyA5_8A2Gs/ParkinsonResearch/runs/baselines_shared_dataset_v1/focal/checkpoints --checkpoint-every 5 --progress --progress-json --pretrained --resume
Device=cuda | fold=5/25 | architecture=wide_resnet50_2 | freeze_policy=early | pretrained=True
Label source=consensus | classes=('control', 'parkinson', '

... (14 outputs left out)


TLO fold 22/25 | loss=focal | freeze=early | seed=42
Command: /usr/bin/python3 -u /content/ParkinsonAiResearch/train.py --fold-id 22 --architecture wide_resnet50_2 --label-source consensus --loss focal --focal-gamma 2.0 --focal-alpha-mode uniform --augmentation standard --rotation-degrees 10.0 --hflip-prob 0.5 --noise-std 0.01 --sampling class_balanced --freeze-policy early --epochs 15 --batch-size 8 --image-size 224 --num-workers 0 --lr 0.001 --backbone-lr 1e-05 --weight-decay 0.0001 --seed 42 --device cuda --manifest /content/ParkinsonData/datasets/annotations/fold_manifest.csv --root /content/ParkinsonData --checkpoint-dir /content/drive/.shortcut-targets-by-id/1UGKWBfKn2fvGjBr39P1v9ndyA5_8A2Gs/ParkinsonResearch/runs/baselines_shared_dataset_v1/focal/checkpoints --checkpoint-every 5 --progress --progress-json --pretrained --resume
Device=cuda | fold=22/25 | architecture=wide_resnet50_2 | freeze_policy=early | pretrained=True
Label source=consensus | classes=('control', 'parkinson',


TLO fold 23/25 | loss=focal | freeze=early | seed=42
Command: /usr/bin/python3 -u /content/ParkinsonAiResearch/train.py --fold-id 23 --architecture wide_resnet50_2 --label-source consensus --loss focal --focal-gamma 2.0 --focal-alpha-mode uniform --augmentation standard --rotation-degrees 10.0 --hflip-prob 0.5 --noise-std 0.01 --sampling class_balanced --freeze-policy early --epochs 15 --batch-size 8 --image-size 224 --num-workers 0 --lr 0.001 --backbone-lr 1e-05 --weight-decay 0.0001 --seed 42 --device cuda --manifest /content/ParkinsonData/datasets/annotations/fold_manifest.csv --root /content/ParkinsonData --checkpoint-dir /content/drive/.shortcut-targets-by-id/1UGKWBfKn2fvGjBr39P1v9ndyA5_8A2Gs/ParkinsonResearch/runs/baselines_shared_dataset_v1/focal/checkpoints --checkpoint-every 5 --progress --progress-json --pretrained --resume
Device=cuda | fold=23/25 | architecture=wide_resnet50_2 | freeze_policy=early | pretrained=True
Label source=consensus | classes=('control', 'parkinson',


TLO fold 24/25 | loss=focal | freeze=early | seed=42
Command: /usr/bin/python3 -u /content/ParkinsonAiResearch/train.py --fold-id 24 --architecture wide_resnet50_2 --label-source consensus --loss focal --focal-gamma 2.0 --focal-alpha-mode uniform --augmentation standard --rotation-degrees 10.0 --hflip-prob 0.5 --noise-std 0.01 --sampling class_balanced --freeze-policy early --epochs 15 --batch-size 8 --image-size 224 --num-workers 0 --lr 0.001 --backbone-lr 1e-05 --weight-decay 0.0001 --seed 42 --device cuda --manifest /content/ParkinsonData/datasets/annotations/fold_manifest.csv --root /content/ParkinsonData --checkpoint-dir /content/drive/.shortcut-targets-by-id/1UGKWBfKn2fvGjBr39P1v9ndyA5_8A2Gs/ParkinsonResearch/runs/baselines_shared_dataset_v1/focal/checkpoints --checkpoint-every 5 --progress --progress-json --pretrained --resume
Device=cuda | fold=24/25 | architecture=wide_resnet50_2 | freeze_policy=early | pretrained=True
Label source=consensus | classes=('control', 'parkinson',


TLO fold 25/25 | loss=focal | freeze=early | seed=42
Command: /usr/bin/python3 -u /content/ParkinsonAiResearch/train.py --fold-id 25 --architecture wide_resnet50_2 --label-source consensus --loss focal --focal-gamma 2.0 --focal-alpha-mode uniform --augmentation standard --rotation-degrees 10.0 --hflip-prob 0.5 --noise-std 0.01 --sampling class_balanced --freeze-policy early --epochs 15 --batch-size 8 --image-size 224 --num-workers 0 --lr 0.001 --backbone-lr 1e-05 --weight-decay 0.0001 --seed 42 --device cuda --manifest /content/ParkinsonData/datasets/annotations/fold_manifest.csv --root /content/ParkinsonData --checkpoint-dir /content/drive/.shortcut-targets-by-id/1UGKWBfKn2fvGjBr39P1v9ndyA5_8A2Gs/ParkinsonResearch/runs/baselines_shared_dataset_v1/focal/checkpoints --checkpoint-every 5 --progress --progress-json --pretrained --resume
Device=cuda | fold=25/25 | architecture=wide_resnet50_2 | freeze_policy=early | pretrained=True
Label source=consensus | classes=('control', 'parkinson',

## 3. Stage 2 — train DeepRBF on the frozen Focal-trained features

The trainer finds the matching Focal checkpoint for every requested fold and verifies the fold ID, class mapping, 2048-D feature width, and exact manifest hash.

The WideResNet is frozen and used only to extract features. DeepRBF is trained independently; Focal Loss is not applied a second time. `--resume` continues from the last saved DeepRBF checkpoint after a Colab runtime reset.

In [8]:
for fold in range(FOLD_START, FOLD_END + 1):
    fold_name = f"fold{fold:02d}_{RBF_DISTANCE}_{RBF_DISTANCE_POWER}"
    fold_dir = RBF_ROOT / fold_name
    bundle_path = fold_dir / "combined_model.pt"
    if bundle_path.is_file():
        print(f"Fold {fold:02d}: combined bundle already exists; skip Stage 2.")
        continue

    baseline_matches = [
        p for p in BASELINE_DIR.glob("*_last.pt")
        if re.search(rf"_fold{fold:02d}_", p.name)
    ]
    if len(baseline_matches) != 1:
        raise RuntimeError(
            f"Fold {fold:02d}: expected one matching Focal baseline before DeepRBF training; "
            f"found {baseline_matches}."
        )

    rbf_command = [
        sys.executable, "-u", "train_deep_rbf_all_folds.py",
        "--baseline-dir", str(BASELINE_DIR), "--baseline-glob", "*_last.pt",
        "--manifest", str(MANIFEST), "--root", str(DATA_ROOT),
        "--output-root", str(RBF_ROOT), "--fold-start", str(fold), "--fold-end", str(fold),
        "--epochs", str(RBF_EPOCHS), "--rbf-dim", str(RBF_DIM),
        "--distance", RBF_DISTANCE, "--distance-power", RBF_DISTANCE_POWER,
        "--known-margin", str(KNOWN_MARGIN), "--reject-margin", str(REJECT_MARGIN),
        "--beta", str(BETA), "--balance-groups", "--lr", str(RBF_LR),
        "--weight-decay", str(RBF_WEIGHT_DECAY), "--seed", str(SEED),
        "--device", DEVICE, "--feature-batch-size", str(FEATURE_BATCH_SIZE), "--resume",
    ]
    print("\nTraining/reusing DeepRBF fold", fold)
    print(" ".join(rbf_command))
    subprocess.run(rbf_command, cwd=PROJECT_DIR, check=True)

print("Stage 2 complete for selected folds.")



Training/reusing DeepRBF fold 1
/usr/bin/python3 -u train_deep_rbf_all_folds.py --baseline-dir /content/drive/.shortcut-targets-by-id/1UGKWBfKn2fvGjBr39P1v9ndyA5_8A2Gs/ParkinsonResearch/runs/baselines_shared_dataset_v1/focal/checkpoints --baseline-glob *_last.pt --manifest /content/ParkinsonData/datasets/annotations/fold_manifest.csv --root /content/ParkinsonData --output-root /content/drive/.shortcut-targets-by-id/1UGKWBfKn2fvGjBr39P1v9ndyA5_8A2Gs/ParkinsonResearch/runs/deeprbf_focal_shared_dataset_v1 --fold-start 1 --fold-end 1 --epochs 30 --rbf-dim 64 --distance l1 --distance-power norm --known-margin 1.0 --reject-margin 1.0 --beta 1.0 --balance-groups --lr 0.001 --weight-decay 0.0 --seed 42 --device cuda --feature-batch-size 8 --resume

Training/reusing DeepRBF fold 2
/usr/bin/python3 -u train_deep_rbf_all_folds.py --baseline-dir /content/drive/.shortcut-targets-by-id/1UGKWBfKn2fvGjBr39P1v9ndyA5_8A2Gs/ParkinsonResearch/runs/baselines_shared_dataset_v1/focal/checkpoints --baseline-

## 4. Stage 3A — select rejection thresholds using development validation only

The tuner prefers a feasible threshold selected from each fold's own validation data. Fold 22 has no validation images, so it is skipped for local fitting and receives the pooled out-of-fold development-validation fallback if that fallback is feasible.

This step does not load the locked test partition. A one-fold pilot may not contain enough validation participants for a feasible threshold; if so, the notebook preserves the trained weights and defers bundle export until more development folds are available.

In [9]:
decision_paths = [
    RBF_ROOT / f"fold{fold:02d}_{RBF_DISTANCE}_{RBF_DISTANCE_POWER}"
    / "threshold_selection" / "rbf_decision.json"
    for fold in range(FOLD_START, FOLD_END + 1)
]
threshold_report_path = RBF_ROOT / "threshold_selection_global" / "threshold_search_report.json"

THRESHOLDS_READY = False
if threshold_report_path.is_file() and all(path.is_file() for path in decision_paths):
    existing_report = json.loads(threshold_report_path.read_text(encoding="utf-8"))
    if existing_report.get("status") == "selected":
        THRESHOLDS_READY = True
        print("Reusing the existing successful validation-only threshold selection.")

if not THRESHOLDS_READY:
    threshold_command = [
        sys.executable, "-u", "tune_deep_rbf_thresholds_all_folds.py",
        "--baseline-dir", str(BASELINE_DIR), "--baseline-glob", "*_last.pt",
        "--deep-rbf-root", str(RBF_ROOT), "--manifest", str(MANIFEST),
        "--root", str(DATA_ROOT), "--fold-start", str(FOLD_START),
        "--fold-end", str(FOLD_END), "--distance", RBF_DISTANCE,
        "--distance-power", RBF_DISTANCE_POWER, "--selection-unit", "participants",
        "--max-pd-to-control-rate", "0.0", "--min-coverage", "0.0",
        "--feature-batch-size", str(FEATURE_BATCH_SIZE), "--device", DEVICE,
    ]
    if threshold_report_path.exists():
        threshold_report_path.unlink()
    print(" ".join(threshold_command))
    threshold_run = subprocess.run(threshold_command, cwd=PROJECT_DIR, check=False)
    if not threshold_report_path.is_file():
        raise RuntimeError(
            f"Threshold tuner failed before writing its report (exit={threshold_run.returncode})."
        )
    threshold_report = json.loads(threshold_report_path.read_text(encoding="utf-8"))
    THRESHOLDS_READY = threshold_report.get("status") == "selected"

if not THRESHOLDS_READY:
    print(
        "No feasible threshold was found for this fold selection. The trained WideResNet "
        "and DeepRBF checkpoints are preserved in Drive. Do not invent a threshold; "
        "train additional development folds and rerun this cell."
    )
else:
    print("Threshold selection complete. Fold-specific decision files are ready.")


/usr/bin/python3 -u tune_deep_rbf_thresholds_all_folds.py --baseline-dir /content/drive/.shortcut-targets-by-id/1UGKWBfKn2fvGjBr39P1v9ndyA5_8A2Gs/ParkinsonResearch/runs/baselines_shared_dataset_v1/focal/checkpoints --baseline-glob *_last.pt --deep-rbf-root /content/drive/.shortcut-targets-by-id/1UGKWBfKn2fvGjBr39P1v9ndyA5_8A2Gs/ParkinsonResearch/runs/deeprbf_focal_shared_dataset_v1 --manifest /content/ParkinsonData/datasets/annotations/fold_manifest.csv --root /content/ParkinsonData --fold-start 1 --fold-end 25 --distance l1 --distance-power norm --selection-unit participants --max-pd-to-control-rate 0.0 --min-coverage 0.0 --feature-batch-size 8 --device cuda
Threshold selection complete. Fold-specific decision files are ready.


## 5. Stage 3B — export combined bundles and evaluate validation (when thresholds are feasible)

Each bundle contains:
- the trained Focal WideResNet weights and preprocessing metadata;
- the separately trained DeepRBF weights and configuration;
- the matching validation-selected rejection thresholds;
- provenance hashes tying the stages together.

The combined model uses `WideResNet.forward_features(images) → DeepRBF(features)`. The baseline classifier logits are not averaged with RBF distances.

In [11]:
if THRESHOLDS_READY:
    from model.deep_rbf_system import load_deep_rbf_bundle

    def find_baseline_checkpoint(fold_id):
        matches = [
            p for p in BASELINE_DIR.glob("*_last.pt")
            if re.search(rf"_fold{fold_id:02d}_", p.name)
        ]
        if len(matches) > 1:
            raise RuntimeError(f"Multiple baseline checkpoints for fold {fold_id}: {matches}")
        return matches[0] if matches else None

    def save_rbf_history(rbf_checkpoint, fold_dir, fold):
        if not rbf_checkpoint.is_file():
            return
        history_path = fold_dir / "training_history.json"
        if history_path.is_file():
            return
        payload = torch.load(rbf_checkpoint, map_location="cpu", weights_only=True)
        history_record = {
            "format_version": "deep-rbf-training-history-v1",
            "fold_id": int(payload["source_baseline"]["fold_id"]),
            "epoch": int(payload.get("epoch", 0)),
            "epochs_requested": int(payload.get("epochs_requested", RBF_EPOCHS)),
            "config": payload.get("config", {}),
            "history": payload.get("history", []),
            "source_baseline": {
                "sha256": payload.get("source_baseline", {}).get("sha256"),
                "fold_manifest_sha256": payload.get("source_baseline", {}).get("fold_manifest_sha256"),
                "fold_id": payload.get("source_baseline", {}).get("fold_id"),
                "baseline_epoch": payload.get("source_baseline", {}).get("baseline_epoch"),
            },
        }
        history_path.write_text(
            json.dumps(history_record, ensure_ascii=False, indent=2) + "\n",
            encoding="utf-8",
        )
        del payload

    for fold in range(FOLD_START, FOLD_END + 1):
        fold_name = f"fold{fold:02d}_{RBF_DISTANCE}_{RBF_DISTANCE_POWER}"
        fold_dir = RBF_ROOT / fold_name
        rbf_checkpoint = fold_dir / "last.pt"
        decision_config = fold_dir / "threshold_selection" / "rbf_decision.json"
        bundle_path = fold_dir / "combined_model.pt"
        validation_dir = fold_dir / "combined_validation"
        validation_report = validation_dir / "val_report.json"

        # Reuse a previously completed bundle only after checking its report/provenance.
        if bundle_path.is_file() and validation_report.is_file():
            loaded_system, loaded_bundle = load_deep_rbf_bundle(
                bundle_path, device=TORCH_DEVICE
            )
            if int(loaded_bundle.get("fold_id", -1)) != fold:
                raise RuntimeError(f"Existing combined bundle has the wrong fold ID: {bundle_path}")

            report = json.loads(validation_report.read_text(encoding="utf-8"))
            if report.get("test_partition_loaded_or_evaluated") is not False:
                raise RuntimeError(f"Validation report unexpectedly indicates test access: {validation_report}")
            if report.get("baseline_sha256") != loaded_bundle["baseline"]["checkpoint_sha256"]:
                raise RuntimeError("Existing bundle and validation report baseline hashes differ.")
            if report.get("stage_checkpoint_sha256") != loaded_bundle["deep_rbf"]["stage_checkpoint_sha256"]:
                raise RuntimeError("Existing bundle and validation report DeepRBF hashes differ.")

            baseline_path = find_baseline_checkpoint(fold)
            if baseline_path is not None:
                if sha256_file(baseline_path) != loaded_bundle["baseline"]["checkpoint_sha256"]:
                    raise RuntimeError("Existing bundle does not match the current baseline checkpoint.")
            if rbf_checkpoint.is_file():
                if sha256_file(rbf_checkpoint) != loaded_bundle["deep_rbf"]["stage_checkpoint_sha256"]:
                    raise RuntimeError("Existing bundle does not match the current DeepRBF checkpoint.")

            del loaded_system, loaded_bundle
            save_rbf_history(rbf_checkpoint, fold_dir, fold)

            if DELETE_STAGE_CHECKPOINTS_AFTER_BUNDLE:
                source_files = [
                    *BASELINE_DIR.glob(f"*_fold{fold:02d}_*_last.pt"),
                    rbf_checkpoint,
                ]
                for source_file in source_files:
                    if source_file.is_file():
                        source_file.unlink()
            print(f"Fold {fold:02d}: existing bundle and validation report verified.")
            continue

        if not rbf_checkpoint.is_file():
            raise FileNotFoundError(
                f"DeepRBF checkpoint missing for fold {fold}: {rbf_checkpoint}"
            )
        baseline_path = find_baseline_checkpoint(fold)
        if baseline_path is None:
            raise FileNotFoundError(
                f"Matching WideResNet checkpoint missing for fold {fold}."
            )
        if not decision_config.is_file():
            raise FileNotFoundError(
                f"Validation-selected decision config missing: {decision_config}"
            )


        rbf_payload = torch.load(
    rbf_checkpoint,
    map_location="cpu",
    weights_only=True
)

# Correct location of fold_id in DeepRBF checkpoint
        source_meta = rbf_payload.get("source_baseline", {})
        checkpoint_fold_id = source_meta.get("fold_id")

        if type(checkpoint_fold_id) is not int or checkpoint_fold_id != fold:
            raise RuntimeError(
        f"DeepRBF checkpoint fold mismatch: "
        f"found={checkpoint_fold_id}, expected={fold}, "
        f"path={rbf_checkpoint}"
    )

        if int(rbf_payload.get("epoch", -1)) != RBF_EPOCHS:
            raise RuntimeError(
        f"DeepRBF fold {fold} is at epoch "
        f"{rbf_payload.get('epoch')}; expected {RBF_EPOCHS}."
    )

        if source_meta.get("sha256") != sha256_file(baseline_path):
            raise RuntimeError(
        f"DeepRBF fold {fold} is not bound to the correct WideResNet."
    )

        if source_meta.get("fold_manifest_sha256") != sha256_file(MANIFEST):
            raise RuntimeError(
        f"Fold {fold} manifest hash mismatch."
    )

        print(
    f"Fold {fold:02d}: DeepRBF checkpoint verified | "
    f"Epoch={rbf_payload['epoch']} | "
    f"Baseline fold={checkpoint_fold_id}"
)

        del rbf_payload

        save_rbf_history(rbf_checkpoint, fold_dir, fold)

        export_command = [
            sys.executable, "export_deep_rbf_bundle.py",
            "--deep-rbf-checkpoint", str(rbf_checkpoint),
            "--baseline-checkpoint", str(baseline_path),
            "--decision-config", str(decision_config),
            "--manifest", str(MANIFEST), "--root", str(DATA_ROOT),
            "--output", str(bundle_path), "--device", DEVICE,
        ]
        subprocess.run(export_command, cwd=PROJECT_DIR, check=True)

        # Evaluate only the development validation split before any cleanup.
        validation_command = [
            sys.executable, "eval_deep_rbf.py",
            "--checkpoint", str(rbf_checkpoint),
            "--baseline-checkpoint", str(baseline_path),
            "--manifest", str(MANIFEST), "--root", str(DATA_ROOT),
            "--split", "val", "--decision-config", str(decision_config),
            "--feature-batch-size", str(FEATURE_BATCH_SIZE), "--device", DEVICE,
            "--output-dir", str(validation_dir),
        ]
        subprocess.run(validation_command, cwd=PROJECT_DIR, check=True)

        if not validation_report.is_file():
            raise RuntimeError(f"Validation report was not created: {validation_report}")
        report = json.loads(validation_report.read_text(encoding="utf-8"))
        if report.get("test_partition_loaded_or_evaluated") is not False:
            raise RuntimeError("Validation evaluation unexpectedly accessed the held-out test set.")

        loaded_system, loaded_bundle = load_deep_rbf_bundle(
            bundle_path, device=TORCH_DEVICE
        )
        if int(loaded_bundle.get("fold_id", -1)) != fold:
            raise RuntimeError(f"Exported bundle has the wrong fold ID: {bundle_path}")
        if loaded_bundle["baseline"]["checkpoint_sha256"] != sha256_file(baseline_path):
            raise RuntimeError("Exported bundle baseline hash does not match its source checkpoint.")
        if loaded_bundle["deep_rbf"]["stage_checkpoint_sha256"] != sha256_file(rbf_checkpoint):
            raise RuntimeError("Exported bundle DeepRBF hash does not match its source checkpoint.")
        if report.get("baseline_sha256") != loaded_bundle["baseline"]["checkpoint_sha256"]:
            raise RuntimeError("Validation report baseline provenance does not match the bundle.")
        if report.get("stage_checkpoint_sha256") != loaded_bundle["deep_rbf"]["stage_checkpoint_sha256"]:
            raise RuntimeError("Validation report DeepRBF provenance does not match the bundle.")
        del loaded_system, loaded_bundle

        # Delete duplicates only after bundle loading, hash checks and validation succeed.
        if DELETE_STAGE_CHECKPOINTS_AFTER_BUNDLE:
            baseline_path.unlink()
            rbf_checkpoint.unlink()
            print(f"Fold {fold:02d}: bundle verified; duplicate source .pt files removed.")
        else:
            print(f"Fold {fold:02d}: bundle verified; source checkpoints retained.")

    print("Combined bundles and validation reports are saved under:", RBF_ROOT)
else:
    print(
        "Bundle export/evaluation is deferred: no feasible validation-only threshold "
        "was found. Source checkpoints are preserved."
    )


Fold 01: DeepRBF checkpoint verified | Epoch=30 | Baseline fold=1
Fold 01: bundle verified; duplicate source .pt files removed.
Fold 02: DeepRBF checkpoint verified | Epoch=30 | Baseline fold=2
Fold 02: bundle verified; duplicate source .pt files removed.
Fold 03: DeepRBF checkpoint verified | Epoch=30 | Baseline fold=3
Fold 03: bundle verified; duplicate source .pt files removed.
Fold 04: DeepRBF checkpoint verified | Epoch=30 | Baseline fold=4
Fold 04: bundle verified; duplicate source .pt files removed.
Fold 05: DeepRBF checkpoint verified | Epoch=30 | Baseline fold=5
Fold 05: bundle verified; duplicate source .pt files removed.
Fold 06: DeepRBF checkpoint verified | Epoch=30 | Baseline fold=6
Fold 06: bundle verified; duplicate source .pt files removed.
Fold 07: DeepRBF checkpoint verified | Epoch=30 | Baseline fold=7
Fold 07: bundle verified; duplicate source .pt files removed.
Fold 08: DeepRBF checkpoint verified | Epoch=30 | Baseline fold=8
Fold 08: bundle verified; duplicate so

## 6. Compose the two trained models in Python

This cell demonstrates the actual composed forward pass on one nonempty development validation fold. It reports distances and decisions; it does not use expert or clinical labels as model inputs.

For a full 25-fold run, fold 1 is selected for this demonstration. If you run only fold 22, choose another fold with nonempty validation.

In [12]:
from model.deep_rbf_system import load_deep_rbf_bundle, WideResNetDeepRBF
from utils.rbf_pipeline import load_deep_rbf_checkpoint
from config import PreprocessingConfig
from data.dataloader import SpiralDataset, build_eval_transform
from torch.utils.data import DataLoader

demo_fold = next((f for f in range(FOLD_START, FOLD_END + 1) if f != 22), None)
if demo_fold is None:
    raise ValueError("The selected fold range contains no nonempty validation fold.")

demo_dir = RBF_ROOT / f"fold{demo_fold:02d}_{RBF_DISTANCE}_{RBF_DISTANCE_POWER}"
if THRESHOLDS_READY:
    system, bundle = load_deep_rbf_bundle(
        demo_dir / "combined_model.pt", device=TORCH_DEVICE
    )
    decision = bundle["decision"]
    preprocessing_data = bundle["baseline"]["preprocessing"]
else:
    # Forward-only composition for a pilot whose validation set is too small
    # to select a defensible rejection threshold.
    backbone, rbf_head, rbf_checkpoint = load_deep_rbf_checkpoint(
        demo_dir / "last.pt", MANIFEST, DATA_ROOT, TORCH_DEVICE
    )
    system = WideResNetDeepRBF(backbone, rbf_head).to(TORCH_DEVICE).eval()
    preprocessing_data = rbf_checkpoint["source_baseline"]["preprocessing"]
    decision = None

preprocessing = PreprocessingConfig.from_dict(preprocessing_data["config"])
val_dataset = SpiralDataset(
    manifest_path=MANIFEST,
    root=DATA_ROOT,
    fold_id=demo_fold,
    split="val",
    image_size=preprocessing.image_size,
    label_source="consensus",
    transform=build_eval_transform(preprocessing),
    allow_empty=True,
)
if len(val_dataset) == 0:
    print(f"Fold {demo_fold} has empty validation; select another fold for the demo.")
else:
    val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=0)
    batch = next(iter(val_loader))
    with torch.no_grad():
        distances = system(batch["image"].to(TORCH_DEVICE))
    if decision is not None:
        labels, reasons, distances = system.predict(
            batch["image"].to(TORCH_DEVICE),
            distance_threshold=float(decision["distance_threshold"]),
            gap_threshold=float(decision["gap_threshold"]),
        )
        names = {0: "control", 1: "parkinson", 2: "unknown"}
        for i, drawing_id in enumerate(batch["drawing_id"]):
            print({
                "drawing_id": drawing_id,
                "distance_control": float(distances[i, 0].cpu()),
                "distance_parkinson": float(distances[i, 1].cpu()),
                "prediction": names[int(labels[i].cpu())],
                "reason": reasons[i],
            })
    else:
        print("Combined forward pass succeeded. No threshold was selected on this small pilot.")
        for i, drawing_id in enumerate(batch["drawing_id"]):
            print({
                "drawing_id": drawing_id,
                "distance_control": float(distances[i, 0].cpu()),
                "distance_parkinson": float(distances[i, 1].cpu()),
            })

{'drawing_id': 'control::subject 01::1_subject 01_1_0.png', 'distance_control': 12.367048263549805, 'distance_parkinson': 16.708534240722656, 'prediction': 'unknown', 'reason': 'ambiguous_distance_gap'}
{'drawing_id': 'control::subject 01::1_subject 01_2_0.png', 'distance_control': 7.831226348876953, 'distance_parkinson': 10.985298156738281, 'prediction': 'unknown', 'reason': 'ambiguous_distance_gap'}
{'drawing_id': 'control::subject 01::1_subject 01_3_0.png', 'distance_control': 9.252937316894531, 'distance_parkinson': 9.3310546875, 'prediction': 'unknown', 'reason': 'ambiguous_distance_gap'}
{'drawing_id': 'control::subject 01::1_subject 01_4_0.png', 'distance_control': 12.056550979614258, 'distance_parkinson': 13.37991714477539, 'prediction': 'unknown', 'reason': 'ambiguous_distance_gap'}


## 7. Final checks before any held-out test evaluation

- Confirm each requested fold has a `combined_model.pt` and `combined_validation/val_report.json` if a feasible threshold was selected.
- Fold 22 has no eligible validation images; its validation report should say `empty_validation`. It must not contribute predictions to threshold fitting.
- Confirm the global threshold report lists the expected nonempty validation folds and records `test_partition_loaded_or_evaluated: false`.
- Training histories, baseline validation reports, threshold decisions, and DeepRBF histories remain in Drive.
- With cleanup enabled, source `.pt` checkpoints are removed only after the combined bundle is loaded, provenance hashes match, and validation evaluation succeeds.
- This notebook deliberately stops at development validation. Do not select hyperparameters or thresholds from the locked test partition.


## 8. Run status summary

Run this cell at the end or after reconnecting to Colab. It only inspects files in Drive; it does not train, delete, or evaluate anything.


In [13]:
from pathlib import Path
import json

print("===== DEEPRBF RUN STATUS =====")
threshold_report_path = RBF_ROOT / "threshold_selection_global" / "threshold_search_report.json"
if threshold_report_path.is_file():
    threshold_report = json.loads(threshold_report_path.read_text(encoding="utf-8"))
    print("Global threshold status:", threshold_report.get("status"))
    print("Validation folds used:", threshold_report.get("validation_folds_used"))
    print("Pooled validation images:", threshold_report.get("pooled_validation_images"))
    print("Pooled validation participants:", threshold_report.get("pooled_validation_participants"))
else:
    print("Global threshold report: not created")

rows = []
for fold in range(FOLD_START, FOLD_END + 1):
    fold_dir = RBF_ROOT / f"fold{fold:02d}_{RBF_DISTANCE}_{RBF_DISTANCE_POWER}"
    bundle = fold_dir / "combined_model.pt"
    val_report = fold_dir / "combined_validation" / "val_report.json"
    rbf_checkpoint = fold_dir / "last.pt"
    baseline_matches = [
        p for p in BASELINE_DIR.glob("*_last.pt")
        if re.search(rf"_fold{fold:02d}_", p.name)
    ]
    rows.append({
        "fold": fold,
        "bundle": bundle.is_file(),
        "validation_report": val_report.is_file(),
        "deep_rbf_checkpoint": rbf_checkpoint.is_file(),
        "baseline_checkpoint": bool(baseline_matches),
    })

for row in rows:
    print(row)

print("\nBundles:", sum(r["bundle"] for r in rows), "/", len(rows))
print("Validation reports:", sum(r["validation_report"] for r in rows), "/", len(rows))
print("DeepRBF stage checkpoints retained:", sum(r["deep_rbf_checkpoint"] for r in rows))
print("WideResNet source checkpoints retained:", sum(r["baseline_checkpoint"] for r in rows))
print("Held-out test was not loaded or evaluated by this status cell.")


===== DEEPRBF RUN STATUS =====
Global threshold status: selected
Validation folds used: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 23, 24, 25]
Pooled validation images: 103
Pooled validation participants: 32
{'fold': 1, 'bundle': True, 'validation_report': True, 'deep_rbf_checkpoint': False, 'baseline_checkpoint': False}
{'fold': 2, 'bundle': True, 'validation_report': True, 'deep_rbf_checkpoint': False, 'baseline_checkpoint': False}
{'fold': 3, 'bundle': True, 'validation_report': True, 'deep_rbf_checkpoint': False, 'baseline_checkpoint': False}
{'fold': 4, 'bundle': True, 'validation_report': True, 'deep_rbf_checkpoint': False, 'baseline_checkpoint': False}
{'fold': 5, 'bundle': True, 'validation_report': True, 'deep_rbf_checkpoint': False, 'baseline_checkpoint': False}
{'fold': 6, 'bundle': True, 'validation_report': True, 'deep_rbf_checkpoint': False, 'baseline_checkpoint': False}
{'fold': 7, 'bundle': True, 'validation_report': True, 'deep_rbf_chec

In [14]:

import json
import csv
from pathlib import Path

def read_csv(path):
    if not path.exists():
        return []
    with path.open(encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))

def calculate_metrics(rows, truth_key):
    valid = [
        r for r in rows
        if r.get(truth_key, "").strip().lower()
        in {"control", "parkinson"}
    ]

    n = len(valid)
    if n == 0:
        return {"samples": 0}

    pairs = [
        (
            r[truth_key].strip().lower(),
            r["predicted_label"].strip().lower()
        )
        for r in valid
    ]

    pd_total = sum(y == "parkinson" for y, p in pairs)
    pd_pred = sum(p == "parkinson" for y, p in pairs)
    tp = sum(
        y == "parkinson" and p == "parkinson"
        for y, p in pairs
    )
    pd_to_control = sum(
        y == "parkinson" and p == "control"
        for y, p in pairs
    )

    accepted = sum(
        p in {"control", "parkinson"}
        for y, p in pairs
    )
    correct = sum(y == p for y, p in pairs)

    return {
        "samples": n,
        "accuracy_all_cases": round(correct / n, 4),
        "coverage": round(accepted / n, 4),
        "rejection_rate": round((n-accepted) / n, 4),
        "accepted_accuracy": round(
            sum(y == p and p != "unknown" for y, p in pairs)
            / accepted, 4
        ) if accepted else None,
        "parkinson_precision": round(tp / pd_pred, 4)
            if pd_pred else None,
        "parkinson_recall": round(tp / pd_total, 4)
            if pd_total else None,
        "pd_to_control_errors": pd_to_control,
        "pd_to_control_rate": round(
            pd_to_control / pd_total, 4
        ) if pd_total else None,
    }

image_rows = []
participant_rows = []

print("===== L1 FOLD RESULTS =====")

for fold in range(FOLD_START, FOLD_END + 1):

    folder = (
        RBF_ROOT /
        f"fold{fold:02d}_{RBF_DISTANCE}_{RBF_DISTANCE_POWER}"
    )

    report_path = folder / "combined_validation" / "val_report.json"

    if not report_path.exists():
        print(f"Fold {fold}: REPORT MISSING")
        continue

    report = json.loads(report_path.read_text(encoding="utf-8"))

    decision = report.get("decision_config", {})

    print(f"\nFold {fold:02d} | {report['status']}")
    print(
        "Threshold:",
        decision.get("distance_threshold"),
        "| Gap:",
        decision.get("gap_threshold"),
        "| Source:",
        decision.get("threshold_source")
    )
    print("Image:", report.get("image_level_clinical_metrics"))
    print("Participants:", report.get("participant_level_clinical_metrics"))

    image_rows += read_csv(
        folder / "combined_validation" / "val_image_predictions.csv"
    )

    participant_rows += read_csv(
        folder / "combined_validation" / "val_participant_predictions.csv"
    )

print("\n===== POOLED OUT-OF-FOLD RESULTS =====")

print(
    "IMAGE LEVEL:",
    calculate_metrics(image_rows, "clinical_reference")
)

print(
    "PARTICIPANT LEVEL:",
    calculate_metrics(participant_rows, "true_clinical_label")
)

global_path = (
    RBF_ROOT /
    "threshold_selection_global" /
    "threshold_search_report.json"
)

if global_path.exists():
    global_report = json.loads(global_path.read_text())
    print("\n===== GLOBAL THRESHOLD SEARCH =====")
    print(json.dumps(
        global_report.get("best_candidate"),
        indent=2,
        ensure_ascii=False
    ))


===== L1 FOLD RESULTS =====

Fold 01 | ok
Threshold: 19.16967010498047 | Gap: 7.026527404785156 | Source: pooled_oof_validation_fallback
Image: {'sample_unit': 'images', 'clinical_reference_samples': 4, 'accuracy_all_cases': 0.0, 'coverage': 0.0, 'rejection_rate': 1.0, 'accepted_accuracy': None, 'parkinson_precision': None, 'parkinson_recall_all_cases': None, 'parkinson_to_control_errors': 0, 'parkinson_to_control_rate': None, 'predicted_parkinson': 0}
Participants: {'sample_unit': 'participants', 'clinical_reference_samples': 1, 'accuracy_all_cases': 0.0, 'coverage': 0.0, 'rejection_rate': 1.0, 'accepted_accuracy': None, 'parkinson_precision': None, 'parkinson_recall_all_cases': None, 'parkinson_to_control_errors': 0, 'parkinson_to_control_rate': None, 'predicted_parkinson': 0}

Fold 02 | ok
Threshold: 19.16967010498047 | Gap: 7.026527404785156 | Source: pooled_oof_validation_fallback
Image: {'sample_unit': 'images', 'clinical_reference_samples': 4, 'accuracy_all_cases': 0.0, 'coverag

In [15]:

import json
from pathlib import Path
from statistics import mean

# Use the existing L1 experiment
root = Path(RBF_ROOT)

print("=" * 65)
print("DEEPRBF L1 TRAINING HISTORY")
print("=" * 65)

loss_summary = []
missing_history = []

for fold in range(1, 26):

    fold_dir = root / f"fold{fold:02d}_l1_norm"
    history_path = fold_dir / "training_history.json"

    if not history_path.exists():
        missing_history.append(fold)
        continue

    data = json.loads(history_path.read_text(encoding="utf-8"))
    history = data.get("history", [])

    losses = [
        float(item["training_loss_before_update"])
        for item in history
        if item.get("training_loss_before_update") is not None
    ]

    if not losses:
        print(f"Fold {fold:02d}: No loss history")
        continue

    record = {
        "fold": fold,
        "epochs": len(losses),
        "first_loss": losses[0],
        "last_loss": losses[-1],
        "minimum_loss": min(losses),
    }

    loss_summary.append(record)

    print(
        f"Fold {fold:02d} | "
        f"Epochs={len(losses)} | "
        f"First={losses[0]:.6f} | "
        f"Last={losses[-1]:.6f} | "
        f"Min={min(losses):.6f}"
    )

print("\n===== TRAINING SUMMARY =====")

print("History files:", len(loss_summary))
print("Missing history:", missing_history)

if loss_summary:
    print(
        "Mean initial loss:",
        round(mean(x["first_loss"] for x in loss_summary), 6)
    )
    print(
        "Mean final loss:",
        round(mean(x["last_loss"] for x in loss_summary), 6)
    )
    print(
        "Folds with lower final loss:",
        sum(x["last_loss"] < x["first_loss"] for x in loss_summary),
        "/",
        len(loss_summary)
    )

# Inspect the actual pooled OOF threshold-selection metrics.
report_path = root / "threshold_selection_global" / "threshold_search_report.json"

print("\n===== GLOBAL L1 VALIDATION =====")

if report_path.exists():

    report = json.loads(report_path.read_text(encoding="utf-8"))
    best = report.get("best_candidate") or {}

    print("Status:", report.get("status"))
    print("Validation folds:", report.get("validation_folds_used"))
    print("Validation images:", report.get("pooled_validation_images"))
    print("Validation participants:", report.get("pooled_validation_participants"))

    print("\nSelected distance threshold:", best.get("distance_threshold"))
    print("Selected gap threshold:", best.get("gap_threshold"))

    print("\nParticipant metrics:")
    print(json.dumps(best.get("metrics"), indent=2))

    print("\nImage metrics:")
    print(json.dumps(best.get("image_metrics"), indent=2))

    print("\nParticipant metrics detail:")
    print(json.dumps(best.get("participant_metrics"), indent=2))

    reasons = best.get("decision_reasons", {})
    print("\nDecision reasons:")
    print(json.dumps(reasons, indent=2))

else:
    print("Global threshold report not found:", report_path)


DEEPRBF L1 TRAINING HISTORY
Fold 01 | Epochs=30 | First=19.852844 | Last=1.744499 | Min=1.744499
Fold 02 | Epochs=30 | First=20.104416 | Last=1.919921 | Min=1.919921
Fold 03 | Epochs=30 | First=19.476166 | Last=1.749397 | Min=1.749397
Fold 04 | Epochs=30 | First=18.196449 | Last=1.687712 | Min=1.687712
Fold 05 | Epochs=30 | First=21.094654 | Last=2.028305 | Min=2.028305
Fold 06 | Epochs=30 | First=19.350840 | Last=1.765895 | Min=1.765895
Fold 07 | Epochs=30 | First=19.680445 | Last=1.893263 | Min=1.893263
Fold 08 | Epochs=30 | First=20.577475 | Last=2.021419 | Min=2.017833
Fold 09 | Epochs=30 | First=20.086399 | Last=1.831898 | Min=1.831898
Fold 10 | Epochs=30 | First=20.172050 | Last=1.925901 | Min=1.925901
Fold 11 | Epochs=30 | First=19.544634 | Last=1.806514 | Min=1.806514
Fold 12 | Epochs=30 | First=18.632427 | Last=1.754123 | Min=1.741114
Fold 13 | Epochs=30 | First=20.737507 | Last=1.997458 | Min=1.994054
Fold 14 | Epochs=30 | First=19.939224 | Last=1.879919 | Min=1.879919
Fold 1

In [16]:

import json
import numpy as np
import pandas as pd
from pathlib import Path

root = Path(RBF_ROOT)

# --------------------------------------------------
# 1. Load selected global thresholds
# --------------------------------------------------

report_path = root / "threshold_selection_global" / "threshold_search_report.json"
report = json.loads(report_path.read_text(encoding="utf-8"))

best = report["best_candidate"]

radius = float(best["distance_threshold"])
gap_threshold = float(best["gap_threshold"])

print("=" * 70)
print("L1 DEEP RBF - DISTANCE DIAGNOSTICS")
print("=" * 70)

print("Distance threshold:", radius)
print("Gap threshold:", gap_threshold)
print("Selection unit:", report["selection_unit"])
print("Global image metrics:", best.get("image_metrics"))
print("Global participant metrics:", best.get("participant_metrics"))
print("Decision reasons:", best.get("decision_reasons"))

# --------------------------------------------------
# 2. Load all validation distances
# --------------------------------------------------

frames = []

for fold in range(1, 26):

    path = root / f"fold{fold:02d}_l1_norm" / "val_distances.csv"

    if not path.exists():
        continue

    df = pd.read_csv(path)
    df["fold"] = fold

    frames.append(df)

if not frames:
    raise FileNotFoundError("No validation distance files found")

data = pd.concat(frames, ignore_index=True)

d0 = data["distance_control"].to_numpy(dtype=float)
d1 = data["distance_parkinson"].to_numpy(dtype=float)

labels = data["clinical_label"].astype(str).str.lower().to_numpy()

minimum = np.minimum(d0, d1)
gaps = np.abs(d0 - d1)

# --------------------------------------------------
# 3. Reproduce the exact rejection rule
# --------------------------------------------------

far = minimum > radius

ambiguous = (
    (d0 < radius) &
    (d1 < radius) &
    (gaps < gap_threshold)
)

tied = d0 == d1

rejected = far | ambiguous | tied

predictions = np.where(
    rejected,
    "unknown",
    np.where(d0 < d1, "control", "parkinson")
)

# --------------------------------------------------
# 4. Nearest-class performance WITHOUT rejection
# --------------------------------------------------

known = np.isin(labels, ["control", "parkinson"])

nearest_predictions = np.where(
    d0 < d1, "control", "parkinson"
)

nearest_accuracy = np.mean(
    nearest_predictions[known] == labels[known]
)

print("\n===== RAW CLASS SEPARATION =====")

print("Total validation images:", len(data))
print("Known clinical references:", int(known.sum()))

print(
    "Nearest-distance accuracy:",
    round(float(nearest_accuracy), 4)
)

print("Accepted:", int((~rejected).sum()))
print("Rejected:", int(rejected.sum()))
print("Ambiguous:", int(ambiguous.sum()))
print("Far:", int(far.sum()))
print("Tied:", int(tied.sum()))

# --------------------------------------------------
# 5. Distance distribution
# --------------------------------------------------

print("\n===== GLOBAL DISTANCE DISTRIBUTION =====")

for name, values in [
    ("Control distance", d0),
    ("Parkinson distance", d1),
    ("Minimum distance", minimum),
    ("Absolute distance gap", gaps),
]:

    print("\n", name)

    print(
        pd.Series(values).describe(
            percentiles=[.10, .25, .50, .75, .90, .95, .99]
        ).round(4).to_string()
    )

# --------------------------------------------------
# 6. Class-specific separation
# --------------------------------------------------

print("\n===== CLASS-SPECIFIC ANALYSIS =====")

for label in ["control", "parkinson"]:

    mask = labels == label

    if not mask.any():
        continue

    raw_acc = np.mean(
        nearest_predictions[mask] == label
    )

    print(
        f"\n{label.upper()} | n={int(mask.sum())}"
    )

    print("Nearest-class accuracy:", round(float(raw_acc), 4))
    print("Median gap:", round(float(np.median(gaps[mask])), 4))
    print("Median minimum distance:", round(float(np.median(minimum[mask])), 4))
    print("Accepted:", int((~rejected[mask]).sum()))
    print("Rejected:", int(rejected[mask].sum()))

# --------------------------------------------------
# 7. Fold-specific distance calibration
# --------------------------------------------------

print("\n===== PER-FOLD ANALYSIS =====")

for fold, part in data.groupby("fold"):

    a = part["distance_control"].to_numpy(dtype=float)
    b = part["distance_parkinson"].to_numpy(dtype=float)

    g = np.abs(a - b)
    m = np.minimum(a, b)

    amb = (
        (a < radius) &
        (b < radius) &
        (g < gap_threshold)
    )

    far_fold = m > radius
    tie_fold = a == b

    reject_fold = amb | far_fold | tie_fold

    print(
        f"Fold {int(fold):02d} | "
        f"N={len(part):3d} | "
        f"MedianGap={np.median(g):.4f} | "
        f"MedianMinDist={np.median(m):.4f} | "
        f"Accepted={(~reject_fold).sum():3d} | "
        f"Ambiguous={amb.sum():3d}"
    )

# --------------------------------------------------
# 8. Threshold coverage frontier
# --------------------------------------------------

print("\n===== THRESHOLD COVERAGE FRONTIER =====")

grid_path = root / "threshold_selection_global" / "threshold_grid.csv"
grid = pd.read_csv(grid_path)

feasible = grid[
    grid["feasible"].astype(str).str.lower().isin(["true", "1"])
].copy()

metric_cols = [
    "precision_pd",
    "recall_pd_all_cases",
    "accepted_accuracy",
    "coverage",
    "accuracy_all_cases",
    "pd_to_control_rate",
]

for col in metric_cols:
    feasible[col] = pd.to_numeric(feasible[col], errors="coerce")

for floor in [0.10, 0.20, 0.30, 0.40, 0.50]:

    candidates = feasible[
        feasible["coverage"] >= floor
    ]

    if candidates.empty:
        print(f"\nCoverage >= {floor:.0%}: NO FEASIBLE CANDIDATE")
        continue

    top = candidates.sort_values(
        [
            "precision_pd",
            "recall_pd_all_cases",
            "accepted_accuracy",
            "coverage",
            "accuracy_all_cases",
        ],
        ascending=False
    ).iloc[0]

    print(f"\nCoverage floor: {floor:.0%}")
    print(
        top[
            [
                "distance_threshold",
                "gap_threshold",
                "precision_pd",
                "recall_pd_all_cases",
                "pd_to_control_rate",
                "accepted_accuracy",
                "coverage",
                "accuracy_all_cases",
            ]
        ].to_dict()
    )

print("\n===== DIAGNOSTICS COMPLETED =====")


L1 DEEP RBF - DISTANCE DIAGNOSTICS
Distance threshold: 19.16967010498047
Gap threshold: 7.026527404785156
Selection unit: participants
Global image metrics: {'sample_unit': 'images', 'clinical_reference_samples': 103, 'accuracy_all_cases': 0.019417475728155338, 'coverage': 0.02912621359223301, 'rejection_rate': 0.970873786407767, 'accepted_accuracy': 0.6666666666666666, 'parkinson_precision': 1.0, 'parkinson_recall_all_cases': 0.018867924528301886, 'parkinson_to_control_errors': 1, 'parkinson_to_control_rate': 0.018867924528301886, 'predicted_parkinson': 1}
Global participant metrics: {'sample_unit': 'participants', 'clinical_reference_samples': 32, 'accuracy_all_cases': 0.03125, 'coverage': 0.03125, 'rejection_rate': 0.96875, 'accepted_accuracy': 1.0, 'parkinson_precision': 1.0, 'parkinson_recall_all_cases': 0.05555555555555555, 'parkinson_to_control_errors': 0, 'parkinson_to_control_rate': 0.0, 'predicted_parkinson': 1}
Decision reasons: {'accepted': 3, 'ambiguous_distance_gap': 100}

/tmp/ipykernel_603/2247401994.py:52: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  data = pd.concat(frames, ignore_index=True)


In [17]:

import numpy as np
import pandas as pd
import torch

from pathlib import Path

root = Path(RBF_ROOT)

frames = []

for fold in range(1, 26):
    path = root / f"fold{fold:02d}_l1_norm" / "val_distances.csv"

    if path.exists():
        df = pd.read_csv(path)

        if not df.empty:
            df["fold_id"] = fold
            frames.append(df)

data = pd.concat(frames, ignore_index=True)

def clean(s):
    return s.astype(str).str.strip().str.lower()

data["clinical_label"] = clean(data["clinical_label"])
data["true_expert_label"] = clean(data["true_expert_label"])

d0 = data["distance_control"].to_numpy(float)
d1 = data["distance_parkinson"].to_numpy(float)

data["nearest_prediction"] = np.where(
    d0 < d1, "control", "parkinson"
)

classes = ["control", "parkinson"]

# ------------------------------------------------
# 1. Compare RBF against clinical and expert labels
# ------------------------------------------------

def evaluate(target_column):

    target = data[target_column]
    mask = target.isin(classes)

    y = target[mask]
    pred = data.loc[mask, "nearest_prediction"]

    cm = pd.crosstab(
        y,
        pred,
        rownames=["Actual"],
        colnames=["Predicted"]
    ).reindex(
        index=classes,
        columns=classes,
        fill_value=0
    )

    recalls = []

    for cls in classes:
        support = int((y == cls).sum())

        correct = int(
            ((y == cls) & (pred == cls)).sum()
        )

        recall = correct / support if support else 0
        recalls.append(recall)

    accuracy = float((y == pred).mean())
    balanced_accuracy = float(np.mean(recalls))
    majority = float(y.value_counts(normalize=True).max())

    print("\nTarget:", target_column)
    print("Samples:", len(y))
    print("Class counts:")
    print(y.value_counts().to_dict())
    print("Confusion matrix:")
    print(cm)
    print("Accuracy:", round(accuracy, 4))
    print("Balanced accuracy:", round(balanced_accuracy, 4))
    print("Majority baseline:", round(majority, 4))

evaluate("clinical_label")
evaluate("true_expert_label")

# ------------------------------------------------
# 2. Expert / clinical label agreement
# ------------------------------------------------

valid = (
    data["clinical_label"].isin(classes) &
    data["true_expert_label"].isin(classes)
)

agreement = (
    data.loc[valid, "clinical_label"] ==
    data.loc[valid, "true_expert_label"]
).mean()

print("\n===== LABEL AGREEMENT =====")
print("Samples:", int(valid.sum()))
print("Expert / Clinical agreement:", round(float(agreement), 4))

# ------------------------------------------------
# 3. Signed distance margins
# Positive = nearest distance agrees with target
# ------------------------------------------------

print("\n===== DISTANCE MARGINS =====")

for target_column in ["clinical_label", "true_expert_label"]:

    for cls in classes:

        subset = data[data[target_column] == cls]

        if subset.empty:
            continue

        a = subset["distance_control"].to_numpy(float)
        b = subset["distance_parkinson"].to_numpy(float)

        signed = b - a if cls == "control" else a - b

        print(
            target_column,
            cls,
            "| N:", len(signed),
            "| Median margin:", round(float(np.median(signed)), 4),
            "| Correct nearest ratio:", round(float((signed > 0).mean()), 4)
        )

# ------------------------------------------------
# 4. Inspect four training groups in RBF checkpoints
# ------------------------------------------------

print("\n===== TRAINING GROUP BALANCE =====")

for fold in range(1, 26):

    checkpoint_path = (
        root / f"fold{fold:02d}_l1_norm" / "last.pt"
    )

    if not checkpoint_path.exists():
        continue

    ckpt = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True
    )

    diagnostics = ckpt.get("training_diagnostics", {})

    print(
        f"Fold {fold:02d} | "
        f"Groups: {diagnostics.get('four_group_counts')} | "
        f"Missing: {diagnostics.get('missing_groups')}"
    )

# ------------------------------------------------
# 5. Compare with existing WideResNet validation
# ------------------------------------------------

baseline_dir = (
    root.parent /
    "baselines_shared_dataset_v1" /
    "focal" /
    "validation_predictions"
)

print("\n===== WIDERESNET FOCAL VALIDATION =====")

if baseline_dir.exists():

    baseline_frames = []

    for path in baseline_dir.glob("*_validation_predictions.csv"):

        df = pd.read_csv(path)

        if df.empty or "fold_id" not in df.columns:
            continue

        if "loss" in df.columns:
            if not df["loss"].astype(str).str.lower().eq("focal").all():
                continue

        baseline_frames.append(df)

    if baseline_frames:

        baseline = pd.concat(
            baseline_frames,
            ignore_index=True
        )

        duplicated = baseline.duplicated(
            ["fold_id", "drawing_id"],
            keep=False
        )

        if duplicated.any():

            print(
                "Multiple baseline prediction runs found. "
                "Skipping ambiguous baseline comparison."
            )

        else:

            baseline["predicted_label"] = clean(
                baseline["predicted_label"]
            )

            merged = data.merge(
                baseline[
                    ["fold_id", "drawing_id", "predicted_label"]
                ],
                on=["fold_id", "drawing_id"],
                how="left",
                validate="one_to_one"
            )

            matched = merged["predicted_label"].notna()

            print("Matched validation images:", int(matched.sum()))

            for target_column in [
                "clinical_label",
                "true_expert_label"
            ]:

                mask = (
                    matched &
                    merged[target_column].isin(classes)
                )

                if mask.any():

                    accuracy = (
                        merged.loc[mask, "predicted_label"] ==
                        merged.loc[mask, target_column]
                    ).mean()

                    print(
                        target_column,
                        "WideResNet accuracy:",
                        round(float(accuracy), 4)
                    )

    else:
        print("No focal validation prediction files found.")

else:
    print("Validation prediction directory not found:", baseline_dir)

print("\n===== DIAGNOSTIC COMPLETE =====")



Target: clinical_label
Samples: 103
Class counts:
{'parkinson': 53, 'control': 50}
Confusion matrix:
Predicted  control  parkinson
Actual                       
control         40         10
parkinson       35         18
Accuracy: 0.5631
Balanced accuracy: 0.5698
Majority baseline: 0.5146

Target: true_expert_label
Samples: 73
Class counts:
{'control': 51, 'parkinson': 22}
Confusion matrix:
Predicted  control  parkinson
Actual                       
control         43          8
parkinson       13          9
Accuracy: 0.7123
Balanced accuracy: 0.6261
Majority baseline: 0.6986

===== LABEL AGREEMENT =====
Samples: 73
Expert / Clinical agreement: 0.6164

===== DISTANCE MARGINS =====
clinical_label control | N: 50 | Median margin: 2.1453 | Correct nearest ratio: 0.8
clinical_label parkinson | N: 53 | Median margin: -0.7558 | Correct nearest ratio: 0.3396
true_expert_label control | N: 51 | Median margin: 2.2242 | Correct nearest ratio: 0.8431
true_expert_label parkinson | N: 22 | Median 

In [18]:

import json
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from model.rbf import reject_distances

root = Path(RBF_ROOT)

# --------------------------------------------------
# 1. Load all L1 validation distances and decisions
# --------------------------------------------------

frames = []

for fold in range(1, 26):

    folder = root / f"fold{fold:02d}_l1_norm"

    csv_path = folder / "val_distances.csv"
    decision_path = folder / "threshold_selection" / "rbf_decision.json"

    if not csv_path.exists() or not decision_path.exists():
        continue

    df = pd.read_csv(csv_path)

    if df.empty:
        continue

    decision = json.loads(
        decision_path.read_text(encoding="utf-8")
    )

    distances = torch.tensor(
        df[["distance_control", "distance_parkinson"]].values,
        dtype=torch.float32
    )

    pred, reasons = reject_distances(
        distances,
        distance_threshold=decision["distance_threshold"],
        gap_threshold=decision["gap_threshold"]
    )

    mapping = {
        0: "control",
        1: "parkinson",
        2: "unknown"
    }

    df["rbf_prediction"] = [
        mapping[int(x)] for x in pred.tolist()
    ]

    df["decision_reason"] = reasons
    df["fold_id"] = fold

    frames.append(df)

data = pd.concat(frames, ignore_index=True)

for col in ["clinical_label", "true_expert_label", "rbf_prediction"]:
    data[col] = (
        data[col].astype(str).str.strip().str.lower()
    )

classes = ["control", "parkinson", "unknown"]

# --------------------------------------------------
# 2. Clinical / expert label agreement
# --------------------------------------------------

print("\n========== LABEL AGREEMENT ==========")

print(
    pd.crosstab(
        data["true_expert_label"],
        data["clinical_label"],
        margins=True
    )
)

known = data["true_expert_label"].isin(
    ["control", "parkinson"]
)

agreement = (
    data.loc[known, "true_expert_label"] ==
    data.loc[known, "clinical_label"]
).mean()

print("Known expert samples:", int(known.sum()))
print("Known expert / clinical agreement:", round(float(agreement), 4))

# --------------------------------------------------
# 3. Three-class evaluation
# --------------------------------------------------

def report_three_class(df, target, prediction, title):

    y = df[target]
    p = df[prediction]

    valid = y.isin(classes) & p.isin(classes)
    y, p = y[valid], p[valid]

    cm = pd.crosstab(y, p).reindex(
        index=classes,
        columns=classes,
        fill_value=0
    )

    print("\n==========", title, "==========")
    print("Samples:", len(y))
    print(cm)
    print("Accuracy:", round(float((y == p).mean()), 4))

    for cls in classes:

        tp = int(((y == cls) & (p == cls)).sum())
        support = int((y == cls).sum())
        predicted = int((p == cls).sum())

        precision = tp / predicted if predicted else 0
        recall = tp / support if support else 0

        print(
            cls,
            "| support:", support,
            "| precision:", round(precision, 4),
            "| recall:", round(recall, 4)
        )

report_three_class(
    data,
    "true_expert_label",
    "rbf_prediction",
    "DEEPRBF L1 THREE CLASS"
)

# --------------------------------------------------
# 4. Clinical metrics after rejection
# --------------------------------------------------

def clinical_report(df, prediction, title):

    valid = df["clinical_label"].isin(
        ["control", "parkinson"]
    )

    y = df.loc[valid, "clinical_label"]
    p = df.loc[valid, prediction]

    accepted = p != "unknown"

    pd_mask = y == "parkinson"

    tp = int(((y == "parkinson") & (p == "parkinson")).sum())
    pd_total = int(pd_mask.sum())

    pd_pred = int((p == "parkinson").sum())

    errors = int(
        ((y == "parkinson") & (p == "control")).sum()
    )

    print("\n==========", title, "==========")
    print("Total:", len(y))
    print("Coverage:", round(float(accepted.mean()), 4))
    print("Rejection:", round(float((~accepted).mean()), 4))
    print("Accepted accuracy:",
          round(float((y[accepted] == p[accepted]).mean()), 4)
          if accepted.any() else None)
    print("Parkinson precision:", round(tp / pd_pred, 4) if pd_pred else None)
    print("Parkinson recall:", round(tp / pd_total, 4))
    print("Parkinson -> Control:", errors)
    print("Parkinson -> Unknown:",
          int((pd_mask & (p == "unknown")).sum()))

clinical_report(
    data,
    "rbf_prediction",
    "DEEPRBF L1 CLINICAL AFTER REJECTION"
)

# --------------------------------------------------
# 5. WideResNet three-class comparison
# --------------------------------------------------

baseline_dir = (
    root.parent /
    "baselines_shared_dataset_v1" /
    "focal" /
    "validation_predictions"
)

baseline_frames = []

for path in baseline_dir.glob("*_validation_predictions.csv"):

    df = pd.read_csv(path)

    if not df.empty:
        baseline_frames.append(df)

if baseline_frames:

    baseline = pd.concat(
        baseline_frames,
        ignore_index=True
    )

    if baseline.duplicated(["fold_id", "drawing_id"]).any():

        print("\nWARNING: Multiple baseline runs found.")
        print("Resolve duplicate fold/drawing predictions before comparison.")

    else:

        baseline["predicted_label"] = (
            baseline["predicted_label"]
            .astype(str).str.strip().str.lower()
        )

        merged = data.merge(
            baseline[
                ["fold_id", "drawing_id", "predicted_label"]
            ],
            on=["fold_id", "drawing_id"],
            how="inner",
            validate="one_to_one"
        )

        print("\nMatched images:", len(merged))

        report_three_class(
            merged,
            "true_expert_label",
            "predicted_label",
            "WIDERESNET FOCAL THREE CLASS"
        )

        clinical_report(
            merged,
            "predicted_label",
            "WIDERESNET FOCAL CLINICAL"
        )

else:
    print("No baseline validation predictions found.")

# --------------------------------------------------
# 6. Four training groups across folds
# --------------------------------------------------

print("\n========== FOUR TRAINING GROUPS ==========")

group_rows = []

for fold in range(1, 26):

    path = root / f"fold{fold:02d}_l1_norm" / "last.pt"

    if not path.exists():
        continue

    ckpt = torch.load(
        path,
        map_location="cpu",
        weights_only=True
    )

    diag = ckpt.get("training_diagnostics", {})

    row = {
        "fold": fold,
        **diag.get("four_group_counts", {})
    }

    group_rows.append(row)

groups = pd.DataFrame(group_rows)

print(groups.to_string(index=False))

if not groups.empty:
    print("\nMissing groups per fold:")
    print(
        groups.set_index("fold")
        .drop(columns=[], errors="ignore")
        .isna().sum(axis=1)
    )

# --------------------------------------------------
# 7. Distance separation
# --------------------------------------------------

print("\n========== DISTANCE MARGINS ==========")

for target in ["clinical_label", "true_expert_label"]:

    for cls in ["control", "parkinson"]:

        subset = data[data[target] == cls]

        if subset.empty:
            continue

        d0 = subset["distance_control"].to_numpy(float)
        d1 = subset["distance_parkinson"].to_numpy(float)

        signed = d1 - d0 if cls == "control" else d0 - d1

        print(
            target, cls,
            "| N:", len(signed),
            "| Median:", round(float(np.median(signed)), 4),
            "| Correct nearest:", round(float((signed > 0).mean()), 4)
        )

print("\n========== COMPLETE ==========")



========== LABEL AGREEMENT ==========
clinical_label     control  parkinson  All
true_expert_label                         
control                 31         20   51
parkinson                8         14   22
unknown                 11         19   30
All                     50         53  103
Known expert samples: 73
Known expert / clinical agreement: 0.6164

========== DEEPRBF L1 THREE CLASS ==========
Samples: 103
rbf_prediction     control  parkinson  unknown
true_expert_label                             
control                  7          3       41
parkinson                1          1       20
unknown                  0          1       29
Accuracy: 0.3592
control | support: 51 | precision: 0.875 | recall: 0.1373
parkinson | support: 22 | precision: 0.2 | recall: 0.0455
unknown | support: 30 | precision: 0.3222 | recall: 0.9667

========== DEEPRBF L1 CLINICAL AFTER REJECTION ==========
Total: 103
Coverage: 0.1262
Rejection: 0.8738
Accepted accuracy: 0.7692
Parkinson precision